<a id="top"></a>

<div style="border:3px solid #2ca02c;background:#eaffea;padding:18px 22px;border-radius:12px;margin:10px 0 18px 0;">
  <div style="font-size:44px;font-weight:900;line-height:1.1;">Analyse von Schlaf- und Lebensstilfaktoren</div>

  <div style="margin-top:12px;display:flex;gap:10px;flex-wrap:wrap;">
    <a href="./01-EDA-Hypothesenentwicklung.ipynb" style="display:inline-block;padding:8px 16px;border-radius:999px;background:#fff;color:#1b5e20;font-size:18px;font-weight:700;text-decoration:none;border:1px solid rgba(46,125,50,0.45);">Explorative Datenanalyse</a>
    <span style="display:inline-block;padding:8px 16px;border-radius:999px;background:#2e7d32;color:#fff;font-size:18px;font-weight:800;box-shadow:0 2px 6px rgba(0,0,0,0.10);">Feature Engineering &amp; Modeling</span>
    <a href="./03-Evaluation-Hypothesenpr%C3%BCfung-Dash-Deployment.ipynb" style="display:inline-block;padding:8px 16px;border-radius:999px;background:#fff;color:#1b5e20;font-size:18px;font-weight:700;text-decoration:none;border:1px solid rgba(46,125,50,0.45);">Evaluation &amp; Deployment</a>
  </div>
</div>

<div style="margin:26px 0 14px 0;">
  <div style="font-size:14px;letter-spacing:1px;color:#555;text-transform:uppercase;">
    Notebook 2
  </div>
  <div style="font-size:30px;font-weight:700;margin-top:4px;">
    Feature Engineering, Segmentierung &amp; Modellierung
  </div>
</div>

### Setup & Laden des Clean-Datensatzes

- Bibliotheken laden  
- Laden des bereinigten Datensatzes  
- Konsistente Labels (für Plots & Tabellen)  
- Dateninitialisierungs-Statusbox  


In [7]:
# Dateninitialisierung
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import statsmodels.api as sm

from pathlib import Path
from IPython.display import display, HTML
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, root_mean_squared_error
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, r2_score)
from sklearn.metrics import confusion_matrix
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_curve
from sklearn.metrics import precision_recall_curve

from plotly.subplots import make_subplots



data_path="data/data_clean.csv"
df_clean = pd.read_csv(data_path)

labels = {
    "age": "Age",
    "gender": "Gender",
    "married": "Married",
    "smoking": "Smoking",
    "bmiCategory": "BMI Category",
    "healthRisk": "Health Risk",
    "riskScore": "Health Risk Score",
    "stress": "Stress Level (1–10)",
    "sleepDuration": "Sleep Duration (hours)",
    "sleepQuality": "Sleep Quality (1–10)",
    "alcohol": "Alcohol (drinks/week)",
    "sugar": "Sugar Intake (g/day)",
    "activity": "Physical Activity (min/day)",
    "steps": "Daily Steps",
    "heartRate": "Heart Rate (bpm)",
    "shortSleep": "Short Sleep",
    "lifestyleRisk": "Lifestyle Risk",
    "stressSleepMismatch": "Stress Sleep Missmatch"
}

# Styling & Farben

COLOR_MAP_GENDER = {
    "Female": "#1f77b4",
    "Male":   "#ff7f0e"
}

COLOR_MAP_RISK = {
    "Low":    "#2ca02c",
    "Medium": "#ff7f0e",
    "High":   "#d62728"
}

COLOR_MAP_BMI = {
    "Underweight": "#9467bd",
    "Normal":      "#1f77b4",
    "Overweight":  "#8c564b"
}

def display_with_labels(
    df,
    n=5,
    column_format=None,
    hide_index=False
):
    """
    df: DataFrame
    n: Anzahl Zeilen
    column_format: dict, z.B. {"age": "{:.0f}", "stress": "{:.1f}"}
    hide_index: True → Index wird ausgeblendet
    """

    df_display = df.head(n).rename(columns=labels)

    styler = df_display.style

    # Formatierung pro Spalte
    if column_format:
        format_dict = {
            labels[col]: fmt
            for col, fmt in column_format.items()
            if col in df.columns
        }
        styler = styler.format(format_dict)

    # Index optional ausblenden
    if hide_index:
        styler = styler.hide(axis="index")

    return styler


rows, cols = df_clean.shape
missing_total = int(df_clean.isna().sum().sum())
dup_total = int(df_clean.duplicated().sum())

num_cols = df_clean.select_dtypes(include="number").columns
cat_cols = df_clean.select_dtypes(exclude="number").columns

html = f"""
<div style="
  border: 1px solid #e5e7eb; border-radius: 12px; padding: 18px 20px; margin: 16px 0;
  background: #ffffff; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Arial, sans-serif;">
  <div style="display:flex; align-items:center; justify-content:space-between; gap:12px;">
    <div style="font-size:18px; font-weight:700; color:#111827;">Notebook 2 · Dateninitialisierung</div>
    <div style="font-size:12px; font-weight:600; padding:4px 10px; border-radius:999px;
                color:#065f46; background:#ecfdf5; border:1px solid #a7f3d0;">READY</div>
  </div>
  <div style="margin-top:10px; color:#374151; font-size:14px; line-height:1.6;">
    Der bereinigte Datensatz wurde erfolgreich geladen ({data_path})
    und ist als <code style="background:#f3f4f6; padding:2px 6px; border-radius:6px;">df_clean</code> verfügbar.<br>
    Damit sind alle folgenden Schritte (Feature Engineering, Clustering, Modeling, Evaluation) reproduzierbar ausführbar.
  </div>
  <div style="margin-top:14px; display:grid; grid-template-columns: repeat(4, minmax(0, 1fr)); gap:12px;">
    <div style="border:1px solid #f3f4f6; border-radius:10px; padding:10px 12px; background:#fafafa;">
      <div style="font-size:12px; color:#6b7280; margin-bottom:4px;">Rows</div>
      <div style="font-size:18px; font-weight:700; color:#111827;">{rows}</div>
    </div>
    <div style="border:1px solid #f3f4f6; border-radius:10px; padding:10px 12px; background:#fafafa;">
      <div style="font-size:12px; color:#6b7280; margin-bottom:4px;">Columns</div>
      <div style="font-size:18px; font-weight:700; color:#111827;">{cols}</div>
    </div>
    <div style="border:1px solid #f3f4f6; border-radius:10px; padding:10px 12px; background:#fafafa;">
      <div style="font-size:12px; color:#6b7280; margin-bottom:4px;">Missing (total)</div>
      <div style="font-size:18px; font-weight:700; color:#111827;">{missing_total}</div>
    </div>
    <div style="border:1px solid #f3f4f6; border-radius:10px; padding:10px 12px; background:#fafafa;">
      <div style="font-size:12px; color:#6b7280; margin-bottom:4px;">Duplicates</div>
      <div style="font-size:18px; font-weight:700; color:#111827;">{dup_total}</div>
    </div>
  </div>
  <div style="margin-top:14px; display:flex; gap:10px; flex-wrap:wrap;">
    <div style="font-size:12px; color:#374151; background:#f3f4f6; border:1px solid #e5e7eb; padding:6px 10px; border-radius:999px;">
      Numeric: <b>{len(num_cols)}</b>
    </div>
    <div style="font-size:12px; color:#374151; background:#f3f4f6; border:1px solid #e5e7eb; padding:6px 10px; border-radius:999px;">
      Categorical: <b>{len(cat_cols)}</b>
    </div>
  </div>
</div>
"""
display(HTML(html))


In [8]:
display_with_labels(
    df_clean,
    n=5,
    column_format={
        "age": "{:.0f}",
        "alcohol": "{:.1f}",
        "sugar": "{:.1f}",
        "activity": "{:.1f}",
        "steps": "{:.0f}",
        "stress": "{:.1f}",
        "sleepDuration": "{:.1f}",
        "sleepQuality": "{:.1f}",
        "heartRate": "{:.0f}"
    },
    hide_index=True
)

Age,Gender,Married,Smoking,Alcohol (drinks/week),Sugar Intake (g/day),Physical Activity (min/day),Daily Steps,Stress Level (1–10),Sleep Duration (hours),Sleep Quality (1–10),Heart Rate (bpm),BMI Category,Health Risk,Health Risk Score
52,Female,Yes,No,9.0,58.2,40.8,1000,3.1,8.4,7.3,69,Normal,Low,0
43,Male,Yes,No,5.8,35.7,93.3,11068,1.0,7.3,10.0,62,Normal,Low,0
55,Female,No,No,10.0,69.6,68.3,6221,4.6,9.0,8.0,66,Normal,Low,0
68,Male,Yes,No,12.7,35.8,87.4,11838,2.7,7.3,8.5,68,Normal,Low,0
41,Male,Yes,No,11.3,85.2,65.6,7994,2.9,5.5,8.8,70,Normal,Low,0


## 1. Ziel dieses Notebooks

**Ziel:** Aus den Erkenntnissen aus Notebook 1 eine reproduzierbare Modell-Pipeline ableiten.

Schwerpunkte:
- Datenvorbereitung für Modellierung (Targets, Features, Skalierung)
- Multikollinearität & Redundanz (modellrelevant)
- Feature Engineering (explorativ, aber plausibel)
- Clustering-Hinweise (Lifestyle-Typen)
- Modellierung: Regression (Vorhersage der Schlafqualität) & Klassifikation (Identifikation von hohem Stress)


## 2. Datenvorbereitung und Harmonisierung

### 2.1 Datengrundlage prüfen

In [9]:
# Datengrundlage
num_summary = (
    df_clean
    .select_dtypes(include="number")
    .describe()
    .T
)

styled_num = (
    num_summary
    .style
    .background_gradient(cmap="Blues")
    .format("{:.2f}")
    .set_caption("Numerische Variablen – Lage- und Streuungsmaße")
)

cat_summary = (
    df_clean
    .select_dtypes(exclude="number")
    .describe()
    .T
)

styled_cat = (
    cat_summary
    .style
    .background_gradient(subset=["count", "unique"], cmap="Greens")
    .set_caption("Kategoriale Variablen – Häufigkeitsstruktur")
)

custom_style = """
<style>
caption {
    caption-side: top;
    font-size: 20px;
    font-weight: bold;
    margin-bottom: 8px;
}
</style>
"""

html_combined = f"""
{custom_style}
<div style="display: flex; gap: 40px; align-items: flex-start;">
    <div style="flex: 1;">
        {styled_num.to_html()}
    </div>
    <div style="flex: 1;">
        {styled_cat.to_html()}
    </div>
</div>
"""

display(HTML(html_combined))

,count,mean,std,min,25%,50%,75%,max
age,2000.00,45.69,14.42,5.00,36.00,46.00,55.00,80.00
alcohol,2000.00,6.40,4.45,0.00,2.98,6.10,9.40,22.90
sugar,2000.00,49.77,23.09,5.00,33.60,49.30,65.10,134.40
activity,2000.00,60.98,29.40,0.00,40.90,60.70,80.50,162.90
steps,2000.00,6857.52,3611.87,1000.00,4102.50,6762.50,9248.00,19186.00
stress,2000.00,4.90,1.99,1.00,3.50,4.80,6.30,10.00
sleepDuration,2000.00,6.89,1.33,4.00,5.90,6.90,7.80,12.00
sleepQuality,2000.00,7.03,1.40,2.20,6.10,7.10,8.00,10.00
heartRate,2000.00,71.78,7.16,52.00,67.00,72.00,76.00,124.00
riskScore,2000.00,0.26,0.55,0.00,0.00,0.00,0.00,2.00


### 2.2 Definition der Zielvariablen

Ziel dieses Abschnitts ist die klare Operationalisierung der Modell-Zielgrößen für Regression und Klassifikation.

Es werden zwei unterschiedliche Vorhersageprobleme definiert:

**1. Regressionsziel**  
Vorhersage der Schlafqualität (`sleepQuality`)
- metrische Zielvariable
- Skala 1–10
- kontinuierliches Regressionsproblem

**2. Klassifikationsziel**  
Identifikation von hohem Stress (`stressHigh`).
- binäre Zielvariable
- inhaltlich definierte Schwelle
- Klassifikationsproblem


In [10]:
# Regressions-Ziel
y_reg = df_clean["sleepQuality"]

# Klassifikations-Ziel
stress_threshold = 7
df_clean["stressHigh"] = (df_clean["stress"] >= stress_threshold).astype(int)
y_clf = df_clean["stressHigh"]

display(y_reg.describe())
display(df_clean[["stress", "stressHigh"]].head())

# Klassenverteilung
display(df_clean["stressHigh"].value_counts().to_frame("Count"))
display(df_clean["stressHigh"].value_counts(normalize=True).round(3).to_frame("Share"))

count    2000.000000
mean        7.027500
std         1.399261
min         2.200000
25%         6.100000
50%         7.100000
75%         8.000000
max        10.000000
Name: sleepQuality, dtype: float64

,stress,stressHigh
0,3.1,0
1,1.0,0
2,4.6,0
3,2.7,0
4,2.9,0


,Count
stressHigh,
0,1683
1,317


,Share
stressHigh,
0,0.842
1,0.158


In [11]:
# Regressions-Ziel
y_reg = df_clean["sleepQuality"]

# Klassifikations-Ziel
stress_threshold = 7
df_clean["stressHigh"] = (df_clean["stress"] >= stress_threshold).astype(int)
y_clf = df_clean["stressHigh"]

# Regression
reg_stats = pd.DataFrame(y_reg.describe()).round(2).T
reg_stats = reg_stats.rename(columns={0: 'Wert'})
display(reg_stats)

# Klassifikation
#display(df_clean[["stress", "stressHigh"]].head(10).style.format({'stress': '{:.1f}'}))
sample_size = 30
np.random.seed(66)
sample_df = df_clean[["stress", "stressHigh"]].sample(n=sample_size, random_state=42).copy()
sample_df["stress"] = sample_df["stress"].round(1).sort_index()
display(sample_df.T)

# Klassenverteilung
dist_df = pd.DataFrame({
    'Klasse': ['Normaler Stress (0)', 'Hoher Stress (1)'],
    'Anzahl': [df_clean["stressHigh"].value_counts()[0], df_clean["stressHigh"].value_counts()[1]],
    'Anteil': [f"{df_clean['stressHigh'].value_counts(normalize=True)[0]*100:.1f}%", 
               f"{df_clean['stressHigh'].value_counts(normalize=True)[1]*100:.1f}%"]
})
display(dist_df.style.hide(axis='index'))

,count,mean,std,min,25%,50%,75%,max
sleepQuality,2000.0,7.03,1.4,2.2,6.1,7.1,8.0,10.0


,1860,353,1333,905,1289,1273,938,1731,65,1323,...,674,1664,1981,1083,1922,99,1179,964,792,29
stress,7.2,3.2,7.2,1.0,5.0,4.8,8.6,6.4,4.5,4.0,...,3.1,2.8,7.0,5.0,2.8,3.2,7.2,5.6,1.1,2.2
stressHigh,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0


Klasse,Anzahl,Anteil
Normaler Stress (0),1683,84.2%
Hoher Stress (1),317,15.8%


**Interpretation**

Die Stress-Skala reicht von 1–10. Werte ab 7 repräsentieren:
- deutlich erhöhte subjektive Belastung
- oberen Bereich der Skala
- praktisch interpretierbare Stresszustände

Die Klassenverteilung zeigt, ob ein Ungleichgewicht vorliegt.  
Die Minderheitsklassen ist stark kleiner.  Die Klassenverteilung zeigt eine moderate Imbalance (15.8 % High Stress).
Dies erfordert bei der Modellbewertung die Nutzung geeigneter Metriken (z. B. ROC-AUC, Recall) anstelle reiner Accuracy.

In [12]:
base_features = [
    "stress",
    "sleepDuration",
    "heartRate",
    "activity",
    "alcohol",
    "sugar",
    "age",
]

X = df_clean[base_features].copy()

y_reg = df_clean["sleepQuality"]
y_clf = df_clean["stressHigh"]

display_with_labels(
    X,
    n=5,
    column_format={
        "age": "{:.0f}",
        "alcohol": "{:.1f}",
        "sugar": "{:.1f}",
        "activity": "{:.1f}",
        "age": "{:.0f}",
        "stress": "{:.1f}",
        "sleepDuration": "{:.1f}",
        "heartRate": "{:.0f}"
    },
    hide_index=True
)

Stress Level (1–10),Sleep Duration (hours),Heart Rate (bpm),Physical Activity (min/day),Alcohol (drinks/week),Sugar Intake (g/day),Age
3.1,8.4,69,40.8,9.0,58.2,52
1.0,7.3,62,93.3,5.8,35.7,43
4.6,9.0,66,68.3,10.0,69.6,55
2.7,7.3,68,87.4,12.7,35.8,68
2.9,5.5,70,65.6,11.3,85.2,41


### 2.3 Auswahl initialer Modell-Features

Es wurde ein kompaktes, gut interpretierbaren Feature-Set (ohne Redundanz wie **Steps + Activity** gleichzeitig) gewählt.

**Interpretation:**  

Das initiale Feature-Set deckt beide Hauptdimensionen (Belastung & Aktivität)
sowie relevante Lifestyle-Variablen ab.

### 2.4 Standardisierung der Features (modellvorbereitend)

**Warum?** Viele Modelle (LogReg, SVM, KNN, PCA) sind skalen-sensitiv.  
**Wie?** `StandardScaler` setzt jede Variable auf z-Skala:  z = (x−μ)/σ.

**Wichtig:** Den Scaler nur auf Trainingsdaten fitten (Leakage vermeiden).


In [13]:
# gemeinsamer Split (stratifiziert nach StressHigh)
X_train, X_test, y_train_reg, y_test_reg, y_train_clf, y_test_clf = train_test_split(
    X, y_reg, y_clf,
    test_size=0.2,
    random_state=42,
    stratify=y_clf
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# als DataFrames (Lesbarkeit)
X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=base_features, index=X_train.index)
X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=base_features, index=X_test.index)

X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=X_train.columns)


# Vergleich: Originaldaten vs. skalierte Daten
original_stats = pd.DataFrame({
    "mean (original)": X_train.mean(),
    "std (original)": X_train.std(ddof=0)
})

scaled_stats = pd.DataFrame({
    "mean (scaled)": X_train_scaled_df.mean(),
    "std (scaled)": X_train_scaled_df.std(ddof=0)
})

comparison = pd.DataFrame({
    "mean (original)": X_train.mean(),
    "std (original)": X_train.std(ddof=0),
    "mean (scaled)": X_train_scaled_df.mean(),
    "std (scaled)": X_train_scaled_df.std(ddof=0)
}).rename(index=lambda x: labels.get(x, x))

display(
    comparison.style.format({
        "mean (original)": "{:.2f}",
        "std (original)": "{:.2f}",
        "mean (scaled)": "{:.2e}",
        "std (scaled)": "{:.2f}"
    })
)

,mean (original),std (original),mean (scaled),std (scaled)
Stress Level (1–10),4.92,1.99,-4.88e-17,1.00
Sleep Duration (hours),6.88,1.31,-4.88e-17,1.00
Heart Rate (bpm),71.79,7.25,-3.14e-16,1.00
Physical Activity (min/day),61.11,29.20,-1.15e-16,1.00
Alcohol (drinks/week),6.44,4.48,-9.77e-17,1.00
Sugar Intake (g/day),49.71,23.04,3.05e-16,1.00
Age,45.68,14.22,2.39e-16,1.00


**Interpretation:**

Der StandardScaler transformiert jedes Feature nach: 
$$z = (x − μ) / σ$$
Dabei werden μ (Mittelwert) und σ (Standardabweichung) **nur aus den Trainingsdaten** berechnet.

Die Tabelle zeigt:

- ursprüngliche Mittelwerte und Standardabweichungen
- die Eigenschaften der transformierten Daten

Nach der Transformation gilt:
- Alle Features besitzen denselben Wertebereich (z-Skala).
- Keine Variable dominiert das Modell aufgrund ihrer Größenordnung.
- Regressionskoeffizienten werden vergleichbar interpretierbar.
- Mittelwert der Features ≈ 0  
- Standardabweichung der Features = 1  

Die sehr kleinen Mittelwerte (~10⁻¹⁷) entstehen durch numerische Rundungsfehler der Floating-Point-Arithmetik.

## 3. Prüfung auf Multikollinearität

### 3.1 Korrelationsanalyse (modellrelevant)

Ziel der Korrelationsanalyse ist die Identifikation möglicher **Multikollinearität** zwischen Modell-Features.

Multikollinearität tritt auf, wenn zwei oder mehr Variablen sehr stark miteinander korrelieren.  
Dies kann bei vielen Modellen problematisch sein, insbesondere bei:

- linearer Regression
- logistischer Regression
- regularisierten Modellen

Stark korrelierte Variablen enthalten ähnliche Information und können:

- Modellkoeffizienten instabil machen
- Interpretierbarkeit reduzieren
- Overfitting begünstigen

Die Korrelationsmatrix ermöglicht eine schnelle visuelle Prüfung solcher Zusammenhänge.

In [14]:
# Pearson-Korrelation der Modell-Features
corr = X.corr(method="pearson")

# Labels für bessere Lesbarkeit
corr_labeled = corr.copy()
corr_labeled.index = corr_labeled.index.map(lambda c: labels.get(c, c))
corr_labeled.columns = corr_labeled.columns.map(lambda c: labels.get(c, c))

mask = np.triu(np.ones_like(corr, dtype=bool))
corr_masked = corr.mask(mask)
corr_masked.index = corr_masked.index.map(lambda c: labels.get(c, c))
corr_masked.columns = corr_masked.columns.map(lambda c: labels.get(c, c))

# Visualisierung
fig = px.imshow(
    corr_masked,
    text_auto=".2f",
    aspect="auto",
    template="simple_white",
    title="Pearson-Korrelation der Modell-Features"
)
fig.update_layout(
    title_x=0.5,
    width=900,
    height=600)
fig.show()

**Interpretation**

Die Korrelationsmatrix zeigt die linearen Zusammenhänge zwischen den Modell-Features.

Die Korrelationswerte |r| liegen zwischen:
- < 0.3 → schwacher Zusammenhang
-   0.3 ≤ |r| < 0.7 → moderater Zusammenhang
-  |r| ≥ 0.7 → starker Zusammenhang
Für die Modellierung sind insbesondere sehr hohe Korrelationen relevant.

Besondere Aufmerksamkeit gilt Feature-Paaren mit |r| > 0.8, da diese auf mögliche Multikollinearität hinweisen können.

In diesem Datensatz zeigen die meisten Features nur moderate oder schwache Korrelationen, was grundsätzlich positiv für die Modellierung ist.

### 3.2 Variance Inflation Factor (VIF)

Während die Korrelationsmatrix nur paarweise Zusammenhänge zeigt, misst der Variance Inflation Factor (VIF) die Gesamt-Multikollinearität eines Features gegenüber allen anderen Features im Modell.

Der VIF gibt an, wie stark die Varianz eines Regressionskoeffizienten durch lineare Abhängigkeiten zwischen Features aufgebläht wird.

Features mit hohen VIF-Werten enthalten häufig redundante Information und können die Stabilität linearer Modelle beeinträchtigen.

Der VIF wird mit den standardisierten Trainingsdaten berechnet, um Data Leakage zu vermeiden.

Die VIF-Werte sollen **visuell vergleichbar** gemacht werden, um potenziell redundante Features schnell zu identifizieren.  
Ein Balkendiagramm zeigt sofort, welche Variablen die höchste Multikollinearität aufweisen.

- VIF ≈ 1: keine Multikollinearität
- 1–5: moderat (meist unkritisch)
- \>5: potenziell problematisch
- \>10: stark problematisch

In [15]:
X_vif = sm.add_constant(X_train_scaled_df)  # VIF auf train (skaliert) berechnen

vif = pd.DataFrame({
    "feature": X_train_scaled_df.columns,
    "VIF": [variance_inflation_factor(X_vif.values, i+1) for i in range(len(X_train_scaled_df.columns))]
}).sort_values("VIF", ascending=False)

vif["feature"] = vif["feature"].map(lambda c: labels.get(c, c))
print(vif)

vif_plot = vif.copy()

fig = px.bar(
    vif_plot,
    x="VIF",
    y="feature",
    orientation="h",
    template="simple_white",
    title="Variance Inflation Factor (VIF) – Multikollinearität der Features",
    labels={"feature": "Feature", "VIF": "VIF"}
)

fig.update_layout(
    title_x=0.5,
    width=900,
    height=420,
    yaxis=dict(categoryorder="total ascending")
)

fig.show()

                       feature       VIF
0          Stress Level (1–10)  1.572578
2             Heart Rate (bpm)  1.379570
1       Sleep Duration (hours)  1.231421
6                          Age  1.036543
3  Physical Activity (min/day)  1.026054
4        Alcohol (drinks/week)  1.011424
5         Sugar Intake (g/day)  1.005001


**Interpretation**
- Balken nahe 1 zeigen weitgehend unabhängige Features.
- Ein klarer Peak (z. B. >5) wäre ein Hinweis auf redundante Variablen, die das Modell destabilisieren könnten.
- Diese Werte zeigen, dass kein Feature stark durch die übrigen Features erklärt werden kann.

Damit bestehen keine Hinweise auf relevante Multikollinearität im Feature-Set.

Für die Modellierung bedeutet dies:
- Regressionskoeffizienten bleiben stabil
- Feature-Effekte sind gut interpretierbar
- kein unmittelbarer Bedarf für Feature-Reduktion

## 4. Feature Engineering

Ziel des Feature Engineering ist es, zusätzliche Variablen zu erzeugen, die relevante Muster im Datensatz besser abbilden als einzelne Rohvariablen.

Dabei werden insbesondere:
- Schwellenwerte
- Kombinationen mehrerer Variablen
- Interaktionen zwischen Variablen  

genutzt.

Die neu erzeugten Features sollen zusätzliche Information für die Modellierung liefern, ohne unnötige Redundanz zu erzeugen.


In [16]:
# Feature Engineering Setup

# Short Sleep
if "shortSleep" not in df_clean.columns:
    df_clean["shortSleep"] = (df_clean["sleepDuration"] < 6).astype(int)

# Lifestyle Risk
if "lifestyleRisk" not in df_clean.columns:

    risk_vars = ["alcohol", "sugar", "activity"]

    scaled = StandardScaler().fit_transform(df_clean[risk_vars])
    risk_df = pd.DataFrame(scaled, columns=risk_vars, index=df_clean.index)

    df_clean["lifestyleRisk"] = (
        risk_df["alcohol"]
        + risk_df["sugar"]
        - risk_df["activity"]
    )

# Stress Sleep Mismatch
if "stressSleepMismatch" not in df_clean.columns:

    df_clean["stressSleepMismatch"] = (
        df_clean["stress"] * (8 - df_clean["sleepDuration"])
    )

print("Feature Engineering abgeschlossen")

Feature Engineering abgeschlossen


### 4.1 Short Sleep (Schlafmangel)

Ziel dieses Features ist die Identifikation von Personen mit
potenziellem Schlafmangel.

Viele medizinische Studien definieren weniger als **6 Stunden Schlaf** als kritische Schwelle.

Die Variable `shortSleep` wird daher binär kodiert:

0 = normaler Schlaf  
1 = kurzer Schlaf

In [17]:
df_clean["shortSleep"] = (df_clean["sleepDuration"] < 6).astype(int)

display(
    df_clean[["sleepDuration", "shortSleep"]]
    .rename(columns=lambda c: labels.get(c, c))
    .head()
    .style
    .format({labels["sleepDuration"]: "{:.1f}"})
    .hide(axis="index")
)

Sleep Duration (hours),Short Sleep
8.4,0
7.3,0
9.0,0
7.3,0
5.5,1


**Interpretation**

Das Feature ermöglicht es Modellen, einen möglichen Schwellenwert-Effekt von Schlafdauer zu erkennen.

Während `sleepDuration` eine kontinuierliche Variable ist, erlaubt `shortSleep` eine klare Trennung zwischen normalem Schlaf und potenziellem Schlafdefizit.

In [18]:
fig = px.scatter(
    df_clean,
    x="sleepDuration",
    y="stress",
    color="shortSleep",
    template="simple_white",
    opacity=0.65,
    title="Short Sleep Overlay: Sleep Duration vs Stress",
    labels=labels,
    color_discrete_map={0: "#1f77b4", 1: "#d62728"}  # blau/rot
)

fig.add_vline(
    x=6,
    line_dash="dash",
    line_color="black",
    line_width=2
)

fig.add_vrect(
    x0=df_clean["sleepDuration"].min(),
    x1=6,
    fillcolor="gray",
    opacity=0.12,
    line_width=0
)

fig.add_annotation(
    x=5.7,
    y=1.1,
    xref="x",
    yref="paper",
    text="Schwelle: < 6h",
    showarrow=False
)

fig.update_traces(marker=dict(size=7))
fig.update_layout(title_x=0.5, width=950, height=520)

fig.show()

**Interpretation (Leitfragen)**
- Die Verteilung der `shortSleep = 1` Punkte (gelb) konzentrieren sich eher bei höheren Stresswerten.
- Im Vergleich dazu sind die `shortSleep = 0` Punkte (lilafarben) über das gesamte Spektrum des Stresslevels verteilt.
- Es gibt eine stake Überlappung. Auffälig ist jedoch, dass im Bereich von sehr niedrigem Stress fas nur lilafarbene Punkte zu finden sind.
- Die Visualisierung unterstützt die Hypothese  **kurzer Schlaf ↔ erhöhter Stress** bedingt.

### 4.2 Lifestyle Risk Index

Mehrere Lifestyle-Faktoren wirken häufig gemeinsam auf Gesundheit
und Schlafqualität.

Der Lifestyle Risk Index kombiniert daher drei Variablen:

- Alkoholkonsum
- Zuckeraufnahme
- körperliche Aktivität

Die Variablen werden zunächst standardisiert,
um unterschiedliche Skalen vergleichbar zu machen.

Der Index wird anschließend berechnet als:

LifestyleRisk = Alcohol + Sugar − Activity

Hohe Werte entsprechen einem ungesünderen Lebensstil.

In [19]:
risk_vars = ["alcohol", "sugar", "activity"]

scaled = StandardScaler().fit_transform(df_clean[risk_vars])
risk_df = pd.DataFrame(scaled, columns=risk_vars)

df_clean["lifestyleRisk"] = (
    risk_df["alcohol"]
    + risk_df["sugar"]
    - risk_df["activity"]
) / 3

display(
    df_clean[["alcohol", "sugar", "activity", "lifestyleRisk"]]
    .rename(columns=lambda c: labels.get(c, c))
    .head()
    .style
    .format({
        labels.get("alcohol", "alcohol"): "{:.1f}",
        labels.get("sugar", "sugar"): "{:.1f}",
        labels.get("activity", "activity"): "{:.1f}",
        labels.get("lifestyleRisk", "lifestyleRisk"): "{:.2f}",
    })
    .hide(axis="index")
)

Alcohol (drinks/week),Sugar Intake (g/day),Physical Activity (min/day),Lifestyle Risk
9.0,58.2,40.8,0.55
5.8,35.7,93.3,-0.61
10.0,69.6,68.3,0.47
12.7,35.8,87.4,-0.03
11.3,85.2,65.6,0.83


**Interpretation**

Der Lifestyle Risk Index fasst mehrere gesundheitsrelevante Verhaltensweisen in einer einzigen Variable zusammen.

Der Lifestyle Risk Index ist:
- ein z-Score-basierter Summenindex
- misst relative Lifestyle-Risiken im Vergleich zur Population
- kombiniert mehrere Verhaltensdimensionen

Der Index ist ein kompakter Indikator für ungesunde Lebensstilfaktoren.

### 4.3 Stress–Sleep-Mismatch

Stress und Schlaf stehen in enger Wechselwirkung.

Besonders problematisch kann eine Kombination aus hohem Stress und gleichzeitig geringer Schlafdauer sein.

Das Feature kombiniert beide Faktoren:

StressSleepMismatch = Stress × (8 − SleepDuration)

Dabei wird angenommen, dass etwa 8 Stunden Schlaf ein idealer Referenzwert sind.

In [20]:
df_clean["stressSleepMismatch"] = (
    df_clean["stress"] * (8 - df_clean["sleepDuration"])
)

display(
    df_clean[["stress", "sleepDuration", "stressSleepMismatch"]]
    .rename(columns=lambda c: labels.get(c, c))
    .head()
    .style
    .format({
        labels.get("stress", "stress"): "{:.1f}",
        labels.get("sleepDuration", "sleepDuration"): "{:.1f}",
        labels.get("stressSleepMismatch", "stressSleepMismatch"): "{:.2f}"
    })
    .hide(axis="index")
)

Stress Level (1–10),Sleep Duration (hours),Stress Sleep Missmatch
3.1,8.4,-1.24
1.0,7.3,0.70
4.6,9.0,-4.60
2.7,7.3,1.89
2.9,5.5,7.25


**Interpretation**

Das Feature misst die Kombination aus psychischer Belastung und Schlafdefizit.

Hohe Werte entstehen bei:

- hohem Stress
- gleichzeitig kurzer Schlafdauer

Das Feature bildet daher einen wichtigen Lifestyle-Konflikt zwischen Stressbelastung und Erholungszeit ab.

### 4.4 Übersicht der neuen Features
Im Feature Engineering wurden drei neue Variablen erzeugt.  
Diese erweitern den Datensatz um Interpretationen, die in einzelnen Variablen nicht direkt sichtbar sind.

In [21]:
new_features = [
    "shortSleep",
    "lifestyleRisk",
    "stressSleepMismatch"
]

feature_meanings = {
    "shortSleep": "Kurze Schlafdauer (<6h)",
    "lifestyleRisk": "Risikofaktoren (Zucker/Alkohol/Bewegungsmangel)",
    "stressSleepMismatch": "Missverhältnis zwischen Stress und Schlaf"
}

stats_df = df_clean[new_features].describe().T

stats_df.insert(0, 'Meaning', [
    feature_meanings[idx] for idx in stats_df.index
])

stats_df.index = [labels.get(idx, idx) for idx in stats_df.index]

display(
    stats_df
    .style
    .format({
        "count": "{:.0f}",
        "mean": "{:.2f}",
        "std": "{:.2f}",
        "min": "{:.2f}",
        "25%": "{:.2f}",
        "50%": "{:.2f}",
        "75%": "{:.2f}",
        "max": "{:.2f}"
    })
)

,Meaning,count,mean,std,min,25%,50%,75%,max
Short Sleep,Kurze Schlafdauer (<6h),2000,0.25,0.43,0.00,0.00,0.00,1.00,1.00
Lifestyle Risk,Risikofaktoren (Zucker/Alkohol/Bewegungsmangel),2000,0.00,0.58,-1.86,-0.41,-0.01,0.37,1.91
Stress Sleep Missmatch,Missverhältnis zwischen Stress und Schlaf,2000,6.62,8.17,-28.40,0.72,4.84,11.20,39.00


### 4.5 Verteilung der neuen Features (Visual Check)

Die neuen Features sollen visuell geprüft werden:
- **Häufigkeiten** für binäre Features (`shortSleep`)
- **Verteilungen** für kontinuierliche Kombinationsfeatures (`lifestyleRisk`, `stressSleepMismatch`)

Die Grafiken dienen als Plausibilitäts- und Qualitätscheck:  
Sind die neuen Features sinnvoll skaliert und interpretiertbar?

In [22]:
tmp = df_clean[["lifestyleRisk", "healthRisk"]].dropna().copy()

# Reihenfolge fixieren
tmp["healthRisk"] = pd.Categorical(
    tmp["healthRisk"],
    categories=["Low", "Medium", "High"],
    ordered=True
)

fig = px.violin(
    tmp.sort_values("healthRisk"),
    x="healthRisk",
    y="lifestyleRisk",
    color="healthRisk",
    color_discrete_map=COLOR_MAP_RISK, 
    box=True,
    points="outliers",
    template="simple_white",
    title="Lifestyle Risk – Verteilung nach Health Risk (Density/Violin)",
    labels=labels
)

fig.update_layout(
    title_x=0.5,
    width=950,
    height=420,
    legend_title_text=""
)

fig.update_traces(meanline_visible=True)  # Mittelwert-Linie anzeigen
fig.show()

**Interpretation**  
- Verschiebung: Es ist ein deutlicher Trend erkennbar: Mit steigendem Gesundheitsrisiko verschiebt sich auch die Verteilung des Lebensstil-Risikos nach oben (höhere Werte).
- Streuung: Die Gruppe "Medium" scheint eine etwas größere vertikale Ausdehnung (Streuung) im Kernbereich zu haben als "Low".
- Überlappung: Es gibt signifikante Überlappungen zwischen den Gruppen, was darauf hindeutet, dass das Lebensstil-Risiko allein nicht ausreicht, um das Gesundheitsrisiko perfekt vorherzusagen.


In [23]:
tmp2 = df_clean[["stressSleepMismatch", "healthRisk"]].dropna().copy()
tmp2["healthRisk"] = pd.Categorical(tmp2["healthRisk"], categories=["Low","Medium","High"], ordered=True)

fig = px.violin(
    tmp2.sort_values("healthRisk"),
    x="healthRisk",
    y="stressSleepMismatch",
    color="healthRisk",
    color_discrete_map=COLOR_MAP_RISK,
    box=True,
    points="outliers",
    template="simple_white",
    title="Stress–Sleep Mismatch – Verteilung nach Health Risk (Density/Violin)",
    labels=labels
)

fig.update_layout(title_x=0.5, width=950, height=420, legend_title_text="")
fig.update_traces(meanline_visible=True)
fig.show()

**Interpretation**  
Das Diagramm zeigt den Zusammenhang zwischen dem Health Risk und dem Stress-Sleep Mismatch (Diskrepanz zwischen Stress und Schlaf):
- Die Violine ist unten am breitesten, die Box ist kurz: Die meisten Menschen in der Gruppe Low Risk haben einen sehr geringen Mismatch. Die Werte sind nah beieinander.
- In der Medium Risk Gruppe ist die Violine ist deutlich höher und nach oben gestreckt: Das Risiko steigt, und die Diskrepanz zwischen Stress und Schlaf nimmt zu.
- Für die High Risk Gruppe ist gesamte Violine ist nach oben verschoben; der Median liegt am höchsten: Menschen mit hohem Gesundheitsrisiko haben auch die höchsten Mismatch-Werte. Es gibt hier kaum noch niedrige Werte (nahe 0).

## 5. Clustering – Lifestyle-Typen
Im Notebook 1 wurden zentrale Muster und Zusammenhänge (Stress–Schlaf, Aktivität–Steps, Health-Risk-Strukturen) untersucht.  
Im Notebook 2 wurden bereits modellrelevante Kombinationsfeatures ergänzt (`shortSleep`, `lifestyleRisk`, `stressSleepMismatch`).

Ziel des Kapitels ist die Identifikation und Interpreation natürlicher Lifestyle-Profile im Datensatz, basierend auf Stress, Schlaf, Aktivität und Gesundheitsverhalten.

Die Cluster sollen die Fragen beantworten:  
- Gibt es typische Lifestyle-Kombinationen?
- Unterscheiden sich diese systematisch im Health Risk?
- Lassen sich daraus verhaltensbasierte Segmente ableiten?

### 5.1 Auswahl der Clustering-Features


Für das Clustering sollen Variablen verwendet werden, die **Lifestyle-Verhalten und Physiologie** beschreiben.
Die Zielvariable `healthRisk` darf **nicht** in die Clusterbildung eingehen (sie wird erst zur Interpretation genutzt).

In [24]:
# Features für Clustering (Lifestyle + Physiologie + Kombinationsfeatures)
cluster_features = [
    "stress",
    "sleepDuration",
    "sleepQuality",
    "activity",
    "heartRate",
    "alcohol",
    "sugar",
    "lifestyleRisk",
    "stressSleepMismatch"
]

# Datengrundlage
X_cluster = df_clean[cluster_features].copy()

# X_cluster.shape

**Interpretation**
- Die Featureauswahl bildet Profile ab (nicht Einzelvariablen).
- Kombinationsfeatures (lifestyleRisk, stressSleepMismatch) verstärken Profil-Kontraste.

### 5.2 Skalierung (Clustering-Vorbereitung)
Distanzbasierte Methoden (z. B. K-Means) sind sensitiv gegenüber unterschiedlichen Skalen.
Daher werden alle Features standardisiert (Mean=0, Std=1).

In [25]:
#from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_cluster_scaled = scaler.fit_transform(X_cluster)

**Interpretation**  
- Ohne Skalierung würde z. B. heartRate oder sugar die Distanz dominieren.
- Standardisierung macht Features vergleichbar und stabilisiert die Clusterbildung.

### 5.3 Bestimmung der Clusterzahl (k)
Die Clusterzahl soll nicht willkürlich gesetzt werden. 
Es werden zwei Heuristiken genutzt:
- Elbow (Inertia)
- Silhouette (Trennschärfe)

In [26]:
K_range = range(2, 9)
inertia = []
sil_scores = []

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=20)
    labels_k = km.fit_predict(X_cluster_scaled)
    inertia.append(km.inertia_)
    sil_scores.append(silhouette_score(X_cluster_scaled, labels_k))

fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(
    go.Scatter(
        x=list(K_range), 
        y=inertia,
        name="Inertia",
        mode="lines+markers",
        line=dict(color="royalblue", width=2),
        marker=dict(size=8)
    ),
    secondary_y=False
)

fig.add_trace(
    go.Scatter(
        x=list(K_range), 
        y=sil_scores,
        name="Silhouette Score",
        mode="lines+markers",
        line=dict(color="crimson", width=2, dash="dash"),
        marker=dict(size=8)
    ),
    secondary_y=True
)

fig.update_layout(
    title="Elbow Method & Silhouette Score – Optimale Clusterzahl",
    title_x=0.5,
    width=850,
    height=420,
    template="simple_white",
    hovermode="x unified"
)

fig.update_xaxes(title_text="k (Anzahl Cluster)")

fig.update_yaxes(
    title_text="Inertia",
    secondary_y=False,
    color="royalblue"
)

fig.update_yaxes(
    title_text="Silhouette Score",
    secondary_y=True,
    color="crimson"
)

fig.show()

c:\Users\UFO\anaconda3\envs\data-analytics\Lib\site-packages\threadpoolctl.py:1226: RuntimeWarning: 
Found Intel OpenMP ('libiomp') and LLVM OpenMP ('libomp') loaded at
the same time. Both libraries are known to be incompatible and this
can cause random crashes or deadlocks on Linux when loaded in the
same Python program.
Using threadpoolctl may cause crashes or deadlocks. For more
information and possible workarounds, please see
    https://github.com/joblib/threadpoolctl/blob/master/multiple_openmp.md

  warnings.warn(msg, RuntimeWarning)


**Interpretation**  

- Elbow: Ein Knick bei k=4 könnte auf einen günstigen Wert hindeuten.
- Silhouette: Hier bedeuten höhere Werte bessere Separation, also eher k=3.

Die Wahl von k ist am Ende eine Trade-off-Entscheidung zwischen Interpretierbarkeit und Fit.

### 5.4 K-Means Clustering (finales k)
Erstellung der Clusterlabels und Anreicherung des Datensatzes um eine cluster-Spalte.

In [27]:
k = 3  # <-- nach Elbow/Silhouette

kmeans = KMeans(n_clusters=k, random_state=42, n_init=30)
cluster_labels = kmeans.fit_predict(X_cluster_scaled)

df_cluster = X_cluster.copy()
df_cluster["cluster"] = cluster_labels

df_cluster["cluster"].value_counts().sort_index()

cluster
0    695
1    653
2    652
Name: count, dtype: int64

**Interpretation**

- Jeder Datenpunkt gehört zu genau einen Cluster.
- Die Clustergrößen geben Hinweise auf dominante vs. seltene Lifestyle-Typen.

### 5.5 Visualisierung der Cluster im PCA-Raum
Cluster in einer 2D-Projektion sichtbar machen, um:
- grobe Separierbarkeit,
- Überlappungen,
- potenzielle Segmentstruktur

zu erkennen.

In [28]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_cluster_scaled)

pca_df = pd.DataFrame(X_pca, columns=["PC1", "PC2"])
pca_df["cluster"] = df_cluster["cluster"].values
pca_df["cluster"] = pca_df["cluster"].astype(str)

fig = px.scatter(
    pca_df,
    x="PC1", y="PC2",
    color="cluster",
    template="simple_white",
    title="Cluster im PCA-Raum (2D)"
)
fig.update_layout(title_x=0.5, width=950, height=520, legend_title_text="")
fig.update_traces(marker=dict(size=7, opacity=0.7))
fig.show()

**Interpretation**
- Starke Überlappung → Cluster eher „weiche Typen“ statt klare Segmente.
- PCA dient der Visualisierung, nicht der Clusterbildung (die erfolgt im Originalraum).

In [29]:
# Anteil der Varianz pro Komponente
exp_var = pca.explained_variance_ratio_
print(f"PC1 erklärt: {exp_var[0]:.2%}")
print(f"PC2 erklärt: {exp_var[1]:.2%}")
print(f"Gesamtvarianz (2D): {sum(exp_var):.2%}")

PC1 erklärt: 34.30%
PC2 erklärt: 21.40%
Gesamtvarianz (2D): 55.70%


In [30]:
# Loadings extrahieren
loadings = pd.DataFrame(
    pca.components_.T, 
    columns=['PC1', 'PC2'], 
        index=df_cluster.drop("cluster", axis=1).columns # Hier die Spaltennamen deines Feature-DFs einsetzen
)

# Top 3 Merkmale
print("Wichtigste Merkmale für PC1:")
print(loadings['PC1'].sort_values(ascending=False).head(3))

Wichtigste Merkmale für PC1:
stressSleepMismatch    0.487868
stress                 0.471398
heartRate              0.321160
Name: PC1, dtype: float64


### 5.6 Clusterprofile (inhaltliche Interpretation)
Die Cluster sollen als Lifestyle-Profile interpretierbar werden.  
Dazu werden die Mittelwerte pro Cluster betrachtet.

In [31]:
#print(df_cluster)
cluster_profiles = (
    df_cluster
    .groupby("cluster")[cluster_features]
    .mean()
    .round(2)
)

display(
    cluster_profiles
    .rename(columns=lambda c: labels.get(c, c))
    .style
    .format("{:.2f}")
)

,Stress Level (1–10),Sleep Duration (hours),Sleep Quality (1–10),Physical Activity (min/day),Heart Rate (bpm),Alcohol (drinks/week),Sugar Intake (g/day),Lifestyle Risk,Stress Sleep Missmatch
cluster,,,,,,,,,
0,4.15,7.40,7.35,77.84,70.37,4.01,37.26,-0.55,2.85
1,6.79,5.76,5.96,52.98,76.11,7.35,51.54,0.19,15.13
2,3.80,7.47,7.75,51.01,68.94,8.00,61.34,0.40,2.11


**Analyse der Clusterprofile**

| Cluster | Charakterisierung | Hauptmerkmale |
|:-------:|-------------------|---------------|
| 0 | Hoher Stress / Schlechter Schlaf | Höchster Stresslevel (7.19), geringste Schlafdauer (5.41h) und Qualität (5.79). Hoher Zuckerkonsum. |
| 1 | Aktive / Ausgeglichen | Hohe körperliche Aktivität (70.94 min), moderate Stress- und Schlafwerte. Niedrigstes Lifestyle-Risiko (-0.45). |
| 2 | Hohes Risiko / Bewegungsmangel | Höchster Alkoholkonsum (8.73) und Zuckeraufnahme (64.74). Geringste Aktivität, höchstes Lifestyle-Risiko (0.53). |
| 3 | Gesund / Ausgeruht | Niedrigster Stress (2.88), bester Schlaf (8.09h) und niedrigste Herzfrequenz (66.75). Sehr aktiv. |

**Interpretation der statistischen Kennzahlen**  
Zusätzlich zur Basis-Analyse lassen sich aus den Kennwerten der Tabelle folgende tiefergehende Erkenntnisse ableiten:
- **Stress Sleep Mismatch (Die Diskrepanz)**:  
Dieser Wert ist bei **Cluster 0 (18.42)** massiv erhöht. Es zeigt, dass Stress und Schlafmangel sich hier nicht nur addieren, sondern ein extremes Ungleichgewicht bilden. Im Gegensatz dazu weist **Cluster 3 (-0.26)** einen negativen Wert auf, was bedeutet, dass die Erholung (Schlaf) die Belastung (Stress) vollumfänglich kompensiert.
- **Lifestyle Risk (Das Gesamtrisiko)**:  
Die Werte sind z-standardisiert (Mittelwert 0).
    - **Cluster 2 (0.53)** liegt deutlich über dem Schnitt und ist damit die primäre Zielgruppe für gesundheitliche Interventionen.
    - **Cluster 1 (-0.45)** und **Cluster 3 (-0.34)** liegen im "grünen Bereich", wobei Cluster 1 durch die hohe Aktivität das geringste statistische Risiko trägt.
- **Herzrate & Physische Aktivität**:
    Es besteht eine klare Korrelation: Die aktivsten Gruppen (Cluster 1 & 3 mit >70 min/Tag) haben die niedrigsten Herzfrequenzen (73.67 & 66.75 bpm). Dies deutet auf eine bessere kardiovaskuläre Fitness hin als bei den eher inaktiven Clustern 0 & 2 (~50 min/Tag).
- **Ernährungsfaktoren (Zucker & Alkohol)**:
    Interessanterweise hat Cluster 0 zwar hohen Stress, aber Cluster 2 nutzt Alkohol (8.73) und Zucker (64.74) am stärksten – möglicherweise als Kompensationsmechanismus für einen eher bewegungsarmen Lebensstil ("Genuss-Sedentary").

**Zusammenfassende Interpretation**

- Cluster 0 (**"Die Gestressten"**): Diese Gruppe weist deutliche Anzeichen von Schlafmangel und hohem Stress auf. Der "Stress Sleep Mismatch" ist hier mit 18.42 extrem hoch, was auf eine starke Diskrepanz zwischen Belastung und Erholung hindeutet.
- Cluster 1 (**"Die Ausgeglichenen"**): Ein gesundes Profil mit viel Bewegung und einer guten Balance. Das negative Lifestyle-Risiko zeigt, dass diese Gruppe unter dem Durchschnitt der Risikofaktoren liegt.
- Cluster 2 (**"Die Risikogruppe"**): Trotz passabler Schlafwerte machen die Ernährung (Zucker/Alkohol) und der Bewegungsmangel dieses Profil zum riskantesten für die langfristige Gesundheit.
- Cluster 3 (**"Die aktiven Langschläfer"**): Das "ideale" Profil. Diese Personen sind physisch fit, psychisch entspannt und schlafen ausreichend.

### 5.7 Clusterprofile (Radar Charts)
Radar Charts sind ideal, um Muster über mehrere Dimensionen gleichzeitig zu vergleichen.  
Multivariate Profile können pro Cluster kompakt dargestellt werden.

In [32]:
# Kernvariablen (übersichtlich)
radar_features = [
    "stress",
    "sleepDuration",
    "sleepQuality",
    "activity",
    "heartRate",
]

cluster_labels = {
    0: "Cluster 0: Die Gestressten",
    1: "Cluster 1: Die Ausgeglichenen",
    2: "Cluster 2: Die Risiko-Gruppe",
    3: "Cluster 3: Die aktiven Langschläfer"
}

cluster_means = df_cluster.groupby("cluster")[radar_features].mean()
cluster_sizes = df_cluster["cluster"].value_counts().sort_index()

# Cluster nach Größe sortieren
cluster_order = cluster_sizes.sort_values(ascending=False).index.tolist()
cluster_means = cluster_means.loc[cluster_order]
sorted_clusters = sorted(cluster_order)

# Standardisieren über Cluster hinweg (vergleichbare Skala)
sc = StandardScaler()
radar_z = pd.DataFrame(
    sc.fit_transform(cluster_means),
    columns=radar_features,
    index=cluster_means.index
)

DEFAULT_CLUSTER_COLORS = [
    "#1f77b4", "#ff7f0e", "#2ca02c",
    "#d62728", "#9467bd", "#8c564b"
]
COLOR_MAP_CLUSTER = {c: DEFAULT_CLUSTER_COLORS[i % len(DEFAULT_CLUSTER_COLORS)]
                     for i, c in enumerate(cluster_order)}

categories = [labels.get(c, c) for c in radar_features]
categories_closed = categories + [categories[0]]

r_min = float(np.nanmin(radar_z.values))
r_max = float(np.nanmax(radar_z.values))
pad = 0.4
r_range = [r_min - pad, r_max + pad]

fig = go.Figure()

for c in sorted_clusters:
    vals = radar_z.loc[c].tolist()
    vals_closed = vals + [vals[0]]
    name = f"{cluster_labels[c]} (n={cluster_sizes.loc[c]})"

    fig.add_trace(go.Scatterpolar(
        r=vals_closed,
        theta=categories_closed,
        fill="toself",
        name=name,
        line=dict(color=COLOR_MAP_CLUSTER[c], width=2),
        opacity=0.55
    ))

fig.update_layout(
    template="simple_white",
    title="Lifestyle-Profile der Cluster (Radar, z-standardisierte Mittelwerte)",
    title_x=0.5,
    width=850,
    height=650,
    legend_title_text="",
    polar=dict(
        radialaxis=dict(visible=True, range=r_range, tickfont=dict(size=11)),
        angularaxis=dict(tickfont=dict(size=13))
    )
)

fig.show()

**Interpretation**  
Der Radar-Chart zeigt die Lifestyle-Profile der vier Cluster: 
- Radar-Achsen sind z-standardisierte Mittelwerte  
    -  0 = stellt den Durchschnitt aller Beobachtuungen dar
    - +1 = deutlich über Durchschnitt
    - −1 = deutlich unter Durchschnitt
- Clusterprofile sind besonders aussagekräftig, wenn mehrere Achsen gleichzeitig abweichen.
- Das Diagramm beschreibt Profile (Segmente), aber keine Kausalität.

- Cluster 0 (Rot): **"Die Gestressten"**
Dieses Profil zeigt ein sehr hohes Stresslevel und eine hohe Ruheherzfrequenz.  
Gleichzeitig ist diese Gruppe körperlich am wenigsten aktiv und hat die schlechteste Schlafqualität.  
- Cluster 1 (Blau): **"Die Ausgeglichenen"**  
Dieses Profil liegt sehr nah am Durchschnitt in fast allen Kategorien.  
Ausßerdem sind sie physisch sehr aktiv.
- Cluster 2 (Orange): **"Die Risiko-Gruppe"**  
Auffallend ist die unterdurchschnittliche Schlafdauer und Schlafqualität.  
Stresslevel und körperliche Aktivität liegen im normalen Bereich.  
- Cluster 3 (Grün): **"Die aktiven Langschläfer"**  
Diese Gruppe ist körperlich extrem aktiv (höchster Wert bei Physical Activity).  
Sie schlafen am längsten, haben aber einen niedrigen Stress Level und eine niedrige Herzfrequenz.  

## 6. Exkurs: MapReduce-Prinzip am Beispiel gruppierter Aggregationen

In diesem Kapitel soll MapReduce als Konzept, nicht als technische Notwendigkeit demonstriert werden. Anhand einer einfachen, aber modellrelevanten Aggregation kann das Prinzip **Map → Shuffle → Reduce**  gezeigt werden.  
MapReduce wird lokal (Chunk-Verarbeitung) simuliert und das Ergebnis mit `pandas.groupby()` verglichen.

Warum das dennoch relevant ist?  
Auch wenn der Datensatz klein ist, bietet MapReduce ein wichtiges Denkmodell für skalierbare Datenverarbeitung (z. B. Spark/Hadoop).  
Viele Operationen lassen sich konzeptionell als MapReduce interpretieren.

Pro Cluster wird berechnet:

- `count` (Anzahl Beobachtungen)
- `mean_sleepQuality` (mittlere Schlafqualität)
- `share_stressHigh` (Anteil hoher Stress)
- `mean_healthRiskScore` (mittleres Gesundheitsrisiko)


### 6.1 Theoretische Einordnung (Map → Shuffle → Reduce)

**Map:**  
Jeder Datenchunk wird unabhängig verarbeitet und liefert Teil-Aggregate pro Schlüssel (hier: `cluster`).  
Beispiel: pro Cluster `count`, `sum_sleepQuality`, `sum_stressHigh`, ...

**Shuffle:**  
Teilresultate werden nach Schlüssel zusammengeführt:  
Alle Teil-Aggregate für `cluster=0` werden gesammelt, dann für `cluster=1`, usw.

**Reduce:**  
Die Teil-Aggregate pro Schlüssel werden zu globalen Aggregaten kombiniert (Summen/Counts addieren),  
danach werden daraus Kennzahlen wie Mittelwerte/Anteile berechnet.

### 6.2 Implementierung: MapReduce-Simulation mit Chunks

Lokale Simulation eines MapReduce-Jobs:

1) Datensatz in Chunks zerlegen  
2) **Map:** pro Chunk Teil-Aggregate berechnen  
3) **Reduce:** Teil-Aggregate zusammenführen  
4) finale Kennzahlen berechnen (Mean/Share)

In [33]:
# Einstellungen
KEY_COL = "cluster"  # Gruppierungsschlüssel (aus Kapitel 5)
CHUNK_SIZE = 300     # nur didaktisch: Chunk-Größe (kleiner => mehr "Map"-Tasks)

cols = [KEY_COL, "sleepQuality"]

if "stressHigh" in df_clean.columns:
    cols.append("stressHigh")
if "riskScore" in df_clean.columns:
    cols.append("riskScore")

if "cluster" in df_cluster.columns:
    mr_df = df_cluster[[KEY_COL]].join(df_clean, how="left")[cols].dropna(subset=[KEY_COL, "sleepQuality"]).copy()
else:
    mr_df = df_clean[cols].dropna(subset=[KEY_COL, "sleepQuality"]).copy()

mr_df.shape, mr_df.head()

((2000, 4),
    cluster  sleepQuality  stressHigh  riskScore
 0        2           7.3           0          0
 1        0          10.0           0          0
 2        2           8.0           0          0
 3        2           8.5           0          0
 4        2           8.8           0          0)

**Map-Funktion**  
Pro Chunk berechnen wir Teil-Aggregate je `cluster`:

- `count`
- `sum_sleepQuality`
- `sum_stressHigh`
- `sum_riskScore`

Das sind genau die Größen, die sich im Reduce-Schritt additiv zusammenführen lassen.

In [34]:
def map_chunk(chunk: pd.DataFrame, key_col: str) -> pd.DataFrame:
    """Map-Schritt: Teil-Aggregate pro key (cluster) in einem Chunk."""
    out = chunk.groupby(key_col).agg(
        count=(key_col, "size"),
        sum_sleepQuality=("sleepQuality", "sum"),
    )

    if "stressHigh" in chunk.columns:
        out["sum_stressHigh"] = chunk.groupby(key_col)["stressHigh"].sum()

    if "riskScore" in chunk.columns:
        out["sum_riskScore"] = chunk.groupby(key_col)["riskScore"].sum()

    return out.reset_index()

**Reduce-Funktion** 
Im Reduce-Schritt werden die Teil-Aggregate über alle Chunks nach Schlüssel zusammengeführt:

- `count` wird summiert
- `sum_*` wird summiert
- daraus wird berechnet:
  - `mean_sleepQuality = sum_sleepQuality / count`
  - `share_stressHigh = sum_stressHigh / count`
  - `mean_riskScore = sum_riskScore / count`

In [35]:
def reduce_partials(partials: list[pd.DataFrame], key_col: str) -> pd.DataFrame:
    """Reduce-Schritt: Teil-Aggregate zusammenführen und finale Kennzahlen berechnen."""
    all_partials = pd.concat(partials, ignore_index=True)

    # Shuffle+Reduce: nach key sammeln und Summen bilden
    agg_cols = [c for c in all_partials.columns if c != key_col]
    reduced = (
        all_partials
        .groupby(key_col)[agg_cols]
        .sum()
        .reset_index()
    )

    # Finale Kennzahlen
    reduced["mean_sleepQuality"] = reduced["sum_sleepQuality"] / reduced["count"]

    if "sum_stressHigh" in reduced.columns:
        reduced["share_stressHigh"] = reduced["sum_stressHigh"] / reduced["count"]

    if "sum_riskScore" in reduced.columns:
        reduced["mean_riskScore"] = reduced["sum_riskScore"] / reduced["count"]

    return reduced

**MapReduce ausführen** (Chunk-Verarbeitung)

In [36]:
partials = []
n_chunks = int(np.ceil(len(mr_df) / CHUNK_SIZE))

for i in range(n_chunks):
    chunk = mr_df.iloc[i*CHUNK_SIZE : (i+1)*CHUNK_SIZE]
    partials.append(map_chunk(chunk, KEY_COL))

mr_result = reduce_partials(partials, KEY_COL).sort_values(KEY_COL).reset_index(drop=True)

mr_result

,cluster,count,sum_sleepQuality,sum_stressHigh,sum_riskScore,mean_sleepQuality,share_stressHigh,mean_riskScore
0,0,695,5106.7,25,39,7.347770,0.035971,0.056115
1,1,653,3893.1,283,415,5.961868,0.433384,0.635528
2,2,652,5055.2,9,57,7.753374,0.013804,0.087423


**Interpretation** (MapReduce-Ergebnis)  
- Jede Zeile entspricht einem Cluster (Schlüssel = `cluster`).
- `count` ist die Clustergröße.
- `mean_sleepQuality` ist die durchschnittliche Schlafqualität im Cluster.
- `share_stressHigh` ist der Anteil "hoher Stress" pro Cluster.
- `mean_riskScore` ist der durchschnittliche Risiko-Score pro Cluster.

Dieses Ergebnis entspricht konzeptionell einem verteilten Job:  
Viele Mapper berechnen Teilsummen, ein Reducer kombiniert diese.

### 6.3 Validierung: Vergleich mit pandas groupby()

Es soll gezeigt werden, dass die MapReduce-Simulation **identische Ergebnisse** wie eine direkte `groupby()`-Aggregation liefert.

In [37]:
# groupby-Referenz (Ground Truth)
gb = mr_df.groupby(KEY_COL).agg(
    count=(KEY_COL, "size"),
    mean_sleepQuality=("sleepQuality", "mean")
).reset_index()

if "stressHigh" in mr_df.columns:
    gb["share_stressHigh"] = mr_df.groupby(KEY_COL)["stressHigh"].mean().values

if "riskScore" in mr_df.columns:
    gb["mean_riskScore"] = mr_df.groupby(KEY_COL)["riskScore"].mean().values

# Vergleichstabelle
compare = gb.merge(
    mr_result[[KEY_COL] + [c for c in mr_result.columns if c.startswith("mean_") or c.startswith("share_") or c == "count"]],
    on=KEY_COL,
    suffixes=("_groupby", "_mapreduce")
)

display(compare.style.hide(axis="index"))

cluster,count_groupby,mean_sleepQuality_groupby,share_stressHigh_groupby,mean_riskScore_groupby,count_mapreduce,mean_sleepQuality_mapreduce,share_stressHigh_mapreduce,mean_riskScore_mapreduce
0,695,7.347770,0.035971,0.056115,695,7.347770,0.035971,0.056115
1,653,5.961868,0.433384,0.635528,653,5.961868,0.433384,0.635528
2,652,7.753374,0.013804,0.087423,652,7.753374,0.013804,0.087423


In [38]:
# Numerische Abweichungen prüfen
diff_cols = []
for c in compare.columns:
    if c.endswith("_groupby"):
        base = c.replace("_groupby", "")
        c_mr = base + "_mapreduce"
        if c_mr in compare.columns and base != KEY_COL:
            diff_name = f"diff_{base}"
            compare[diff_name] = compare[c] - compare[c_mr]
            diff_cols.append(diff_name)

display(compare[[KEY_COL] + diff_cols].style.hide(axis="index"))

cluster,diff_count,diff_mean_sleepQuality,diff_share_stressHigh,diff_mean_riskScore
0,0,0.000000,0.000000,0.000000
1,0,0.000000,0.000000,0.000000
2,0,0.000000,0.000000,0.000000


**Interpretation** (Vergleich)
- Es wurden keine Differenzen festgestellt.
- Damit ist validiert, dass die MapReduce-Simulation korrekt aggregiert.
- In der Praxis wäre `groupby()` natürlich die bessere Wahl für kleine Daten.

MapReduce ist hier ein didaktisches Prinzip für Skalierung.

### 6.4 Visualisierung der Aggregation
Die aggregierten Kennzahlen pro Cluster sollen anschaulich dargestellt werden.

In [39]:
plot_df = mr_result.rename(columns={
    "mean_sleepQuality": "Sleep Quality",
    "share_stressHigh": "High Stress Share",
    "mean_riskScore": "Risk Score"
})

plot_df["cluster"] = plot_df["cluster"].astype(str)

fig = px.parallel_coordinates(
    plot_df,
    color="Risk Score",
    dimensions=["Sleep Quality","High Stress Share","Risk Score"],
    template="simple_white",
    title="Clusterprofil – MapReduce Aggregationen"
)

fig.update_layout(
    title_x=0.5,
    width=900,
    height=420
)

fig.show()

**Interpretation**  
Ein Parallel Coordinates Plot zeigt mehrere Variablen gleichzeitig, indem jede Variable eine vertikale Achse bekommt und jede Beobachtung als Linie durch alle Achsen verläuft. In diesem Fall repräsentiert jede Linie ein Cluster und verbindet dessen Werte über alle Dimensionen.

Die Achsen sind:
- Sleep Quality (durchschnittliche Schlafqualität)
- High Stress Share (Anteil Personen mit hohem Stress Level)
- Risk Score (fasst mehrere Lifestyle-Risikofaktoren zusammen)

Die Darstellung verdeutlicht die strukturellen Unterschiede zwischen den Lifestyle-Typen und bestätigt die in der Clusteranalyse identifizierten Segmente.

Zum Beispiel weist das Cluster mit hoher Stressrate gleichzeitig tendenziell niedrigere Schlafqualität und höhere Risikoscores auf und kann als **Lifestyle-Risikogruppe** interpretiert werden.

Diese Darstellung bestätigt, dass die Cluster nicht nur algorithmisch, sondern auch **inhaltlich interpretierbare Lifestyle-Profile** darstellen.


Die MapReduce-Simulation zeigt, dass viele typische Datenoperationen sich konzeptionell in drei Schritten gliedern lassen:

Map  
→ lokale Teilaggregation

Shuffle  
→ Gruppierung nach Schlüssel

Reduce  
→ Zusammenführung der Teilresultate

Für kleine Daten wird dieses Prinzip durch `groupby()` effizient im Arbeitsspeicher umgesetzt.

In Big-Data-Systemen wie **Hadoop** oder **Spark** wird dieselbe Logik verteilt über viele Rechner ausgeführt.

Der Exkurs verdeutlicht damit, wie analytische Methoden aus diesem Projekt auch auf sehr große Datenmengen übertragbar wären.

In [40]:
# Visualisierung des MapReduce-Prinzips
fig = go.Figure()

# Box-Positionen
x_map = 0.15
x_shuffle = 0.50
x_reduce = 0.85
y = 0.5

# --- Boxen
fig.add_shape(type="rect", x0=x_map-0.12, x1=x_map+0.12, y0=y-0.12, y1=y+0.12,
              line=dict(color="black"), fillcolor="#e3f2fd")
fig.add_shape(type="rect", x0=x_shuffle-0.12, x1=x_shuffle+0.12, y0=y-0.12, y1=y+0.12,
              line=dict(color="black"), fillcolor="#fff3e0")
fig.add_shape(type="rect", x0=x_reduce-0.12, x1=x_reduce+0.12, y0=y-0.12, y1=y+0.12,
              line=dict(color="black"), fillcolor="#e8f5e9")

# --- Pfeile
fig.add_annotation(x=x_shuffle-0.15, y=y, ax=x_map+0.15, ay=y,
                   xref="x", yref="y", axref="x", ayref="y",
                   showarrow=True, arrowhead=3, arrowsize=1.2, arrowwidth=2)
fig.add_annotation(x=x_reduce-0.15, y=y, ax=x_shuffle+0.15, ay=y,
                   xref="x", yref="y", axref="x", ayref="y",
                   showarrow=True, arrowhead=3, arrowsize=1.2, arrowwidth=2)

# --- Texte
fig.add_annotation(x=x_map, y=y+0.04, text="<b>MAP</b>", showarrow=False, font=dict(size=18))
fig.add_annotation(x=x_map, y=y-0.05,
                   text="Chunk 1, Chunk 2, ...<br>Teilaggregate je Cluster",
                   showarrow=False, font=dict(size=12))

fig.add_annotation(x=x_shuffle, y=y+0.04, text="<b>SHUFFLE</b>", showarrow=False, font=dict(size=18))
fig.add_annotation(x=x_shuffle, y=y-0.05,
                   text="Gruppierung nach Schlüssel<br>(cluster)",
                   showarrow=False, font=dict(size=12))

fig.add_annotation(x=x_reduce, y=y+0.04, text="<b>REDUCE</b>", showarrow=False, font=dict(size=18))
fig.add_annotation(x=x_reduce, y=y-0.05,
                   text="Zusammenführung der<br>Teilresultate",
                   showarrow=False, font=dict(size=12))

fig.update_layout(
    template="simple_white",
    title="MapReduce-Prinzip am Beispiel gruppierter Aggregationen",
    title_x=0.5,
    width=1000,
    height=320,
    xaxis=dict(visible=False, range=[0, 1]),
    yaxis=dict(visible=False, range=[0, 1]),
    margin=dict(l=20, r=20, t=60, b=20)
)

fig.show()

- **Map:** Jeder Datenchunk wird unabhängig verarbeitet und liefert Teilaggregate.
- **Shuffle:** Diese Teilaggregate werden nach Schlüssel (`cluster`) gesammelt.
- **Reduce:** Die Teilresultate werden kombiniert und zu globalen Kennzahlen zusammengeführt.

Die Visualisierung zeigt, dass gruppierte Aggregationen in großen verteilten Systemen sehr gut skalieren.

## 7. Regression – Vorhersage der Schlafqualität

In diesem Kapitel wird untersucht, wie gut sich die **Schlafqualität** aus Lifestyle- und Gesundheitsmerkmalen vorhersagen lässt.

Es werden zwei Modelltypen verglichen:

- **Lineare Regression** als interpretierbare Baseline.
- **Random Forest Regressor** als leistungsfähiger nichtlinearer Ansatz.

Die Modelle werden anhand mehrerer Kennzahlen bewertet:
- Root Mean Squared Error (RMSE)
- Mean Absolute Error (MAE)
- R² Score (Coefficient of Determination)

### 7.1 Baseline-Modell (Linear Regression)

Die lineare Regression soll als einfacher Referenzwert dienen.

Vorteile:
- gut interpretierbar
- schnell trainierbar
- Referenz für komplexere Modelle

In [41]:
lr = LinearRegression()

lr.fit(X_train_scaled_df, y_train_reg)

pred_lr = lr.predict(X_test_scaled_df)

#rmse_lr = np.sqrt(mean_squared_error(y_test_reg, pred_lr))
rmse_lr = root_mean_squared_error(y_test_reg, pred_lr)
mae_lr = mean_absolute_error(y_test_reg, pred_lr)
r2_lr = r2_score(y_test_reg, pred_lr)

baseline_results = pd.DataFrame({
    "Model": ["Linear Regression"],
    "Root Mean Squared Error": [rmse_lr],
    "Mean Absolute Error": [mae_lr],
    "Coefficient of Determination (R²)": [r2_lr]
})

display(
    baseline_results
    .style
    .format({
        "Root Mean Squared Error": "{:.3f}",
        "Mean Absolute Error": "{:.3f}",
        "Coefficient of Determination (R²)": "{:.3f}"
    })
    .hide(axis="index")
)

Model,Root Mean Squared Error,Mean Absolute Error,Coefficient of Determination (R²)
Linear Regression,1.072,0.850,0.427


### 7.2 Advanced-Modell (Random Forest Regressor)

Random Forest kann **nichtlineare Zusammenhänge** und **Interaktionen zwischen Variablen** besser abbilden als lineare Modelle.

In [42]:
rf = RandomForestRegressor(
    random_state=42,
    n_estimators=300
)

rf.fit(X_train_scaled_df, y_train_reg)

pred_rf = rf.predict(X_test_scaled_df)

rmse_rf = root_mean_squared_error(y_test_reg, pred_rf)
mae_rf = mean_absolute_error(y_test_reg, pred_rf)
r2_rf = r2_score(y_test_reg, pred_rf)
rf_results = pd.DataFrame({
    "Model": ["Random Forest"],
    "Root Mean Squared Error": [rmse_rf],
    "Mean Absolute Error": [mae_rf],
    "Coefficient of Determination (R²)": [r2_rf]
})

display(
    rf_results
    .style
    .format({
        "Root Mean Squared Error": "{:.3f}",
        "Mean Absolute Error": "{:.3f}",
        "Coefficient of Determination (R²)": "{:.3f}"
    })
    .hide(axis="index")
)

Model,Root Mean Squared Error,Mean Absolute Error,Coefficient of Determination (R²)
Random Forest,1.101,0.869,0.395


### 7.3 Modellvergleich

In [43]:
# Vergleichstabelle
model_compare = pd.concat([baseline_results, rf_results], ignore_index=True)

# Für Plotly in Long-Format bringen
plot_df = model_compare.melt(
    id_vars="Model",
    var_name="Metric",
    value_name="Value"
)

fig = px.bar(
    plot_df,
    x="Metric",
    y="Value",
    color="Model",
    barmode="group",
    template="simple_white",
    title="Modellvergleich – Regression auf Sleep Quality"
)

fig.update_traces(
    width=0.2,  # Deutlich schmalere Balken (vorher default ~0.8-0.9)
    marker_line_width=0.5,
    marker_line_color="white"
)

fig.update_layout(
    title_x=0.5,
    width=950,
    height=450,
    xaxis_title="",
    yaxis_title="Wert",
    legend_title_text="",
    bargap=0.5,  # Abstand zwischen Balkengruppen
    bargroupgap=0.5  # Abstand innerhalb der Gruppe
)

fig.show()

**Interpretation**

Die Grafik vergleicht die Regressionsmodelle über drei zentrale Gütemaße:

- **Root Mean Squared Error**: durchschnittlicher Vorhersagefehler mit stärkerer Gewichtung großer Fehler  
- **Mean Absolute Error**: durchschnittliche absolute Abweichung  
- **Coefficient of Determination (R²)**: erklärte Varianz  

Ein besseres Modell zeigt:
- **niedrigere Fehlerwerte** (RMSE, MAE)
- **höheres R²**

Damit lässt sich erkennen, dass das komplexere Modell (Random Forest) in diesem Fall keinen echten Mehrwert gegenüber der linearen Baseline bietet.

### 7.4 Prediction vs Actual

Dieser Plot zeigt, wie gut die Vorhersagen mit den tatsächlichen Werten übereinstimmen.  
Idealerweise liegen die Punkte nahe an der Diagonalen.

In [44]:
plot_df = pd.DataFrame({
    "Actual": y_test_reg,
    "Predicted": pred_rf
})

# Fehler berechnen
plot_df["Error"] = np.abs(plot_df["Actual"] - plot_df["Predicted"])

# gleiche Skala für beide Achsen
axis_min = min(plot_df["Actual"].min(), plot_df["Predicted"].min())
axis_max = max(plot_df["Actual"].max(), plot_df["Predicted"].max())

fig = px.scatter(
    plot_df,
    x="Actual",
    y="Predicted",
    color="Error",
    color_continuous_scale="Turbo",
    template="simple_white",
    opacity=0.7,
    title="Prediction vs Actual – Random Forest"
)

# perfekte Vorhersage
fig.add_shape(
    type="line",
    x0=axis_min,
    y0=axis_min,
    x1=axis_max,
    y1=axis_max,
    line=dict(color="black", dash="dash", width=3)
)

fig.update_layout(
    title_x=0.5,
    width=900,
    height=520,
    xaxis=dict(range=[axis_min, axis_max]),
    yaxis=dict(range=[axis_min, axis_max]),
    xaxis_title="Actual Sleep Quality",
    yaxis_title="Predicted Sleep Quality"
)

fig.show()

**Interpretation der Modellleistung**  
- Allgemeiner Trend: Die Punktwolke folgt grob der diagonalen Linie, was bedeutet, dass das Modell einen Zusammenhang gelernt hat.
- Genauigkeit: Viele Punkte liegen nah an der Linie (dunkelblau), besonders im mittleren Bereich (Schlafqualität 6–8).
- Ausreißer: Es gibt einige deutliche Abweichungen.  
  Zum Beispiel wurde für einen tatsächlichen Wert von ca. 4 eine Qualität von fast 7 vorhergesagt (oranger Punkt oben links). Ebenso wurde ein tatsächlicher Wert von ca. 9,5 nur als 5,5 vorhergesagt (roter Punkt unten rechts).
- Varianz: Das Modell scheint die extremen Werte (sehr niedrige oder sehr hohe Schlafqualität) etwas schlechter zu erfassen als die Durchschnittswerte.

### 7.5 Feature Importance

Random Forest erlaubt eine explorative Bewertung der **relativen Bedeutung der Features**.

In [45]:
imp = pd.Series(
    rf.feature_importances_,
    index=base_features
).sort_values()

imp.index = imp.index.map(lambda c: labels.get(c, c))

fig = px.bar(
    imp.reset_index().rename(columns={"index":"Feature", 0:"Importance"}),
    x="Importance",
    y="Feature",
    orientation="h",
    template="simple_white",
    title="Feature Importance – Random Forest (Regression)"
)

fig.update_layout(
    title_x=0.5,
    width=900,
    height=500
)

fig.show()

#imp

**Interpretation**  
Welche Faktoren dominieren?  
- Dominanter Faktor: Der **Stress Level (1-10)** ist mit einem Wert von ca. 0,549 (54,9 %) der absolut dominierende Faktor. Er trägt mehr zur Entscheidung des Modells bei als alle anderen Merkmale zusammen.
- Sekundäre Faktoren: Mit großem Abstand folgen:
    - Sugar Intake (g/day): 0,087 (~8,7 %)
    - Physical Activity (min/day): 0,082 (~8,2 %)
- Geringere Bedeutung: Faktoren wie das Alter, die Schlafdauer, Alkoholkonsum und die Herzfrequenz liegen alle in einem ähnlichen, niedrigen Bereich zwischen 6 % und 7,5 %.  

Das Modell verlässt sich bei der Vorhersage primär auf das Stresslevel. Wenn man die Zielvariable beeinflussen möchte, wäre Stressmanagement laut diesem Modell der effektivste Hebel.

Wie gut ist die Vorhersage?  

Die Vorhersagequalität des Modells ist eher schwach. Hier ist die detaillierte Einordnung der Werte:
- Bestimmtheitsmaß (R²=0.395):
Das Modell erklärt nur etwa 39,5 % der Varianz in den Daten. Das bedeutet, dass über 60 % der Unterschiede in der Zielvariable durch Faktoren verursacht werden, die das Modell entweder nicht kennt oder nicht korrekt erfassen kann. Ein Wert unter 0,5 gilt als Indikator für eine geringe Vorhersagekraft.  
- Fehlermetriken (RMSE & MAE):  
    - Der Mean Absolute Error (MAE) von 0,869 besagt, dass die Vorhersagen des Modells im Durchschnitt um ca. 0,87 Einheiten vom tatsächlichen Wert abweichen.
    - Der Root Mean Squared Error (RMSE) von 1,072 ist höher als der MAE, was darauf hindeutet, dass es einige Ausreißer gibt, bei denen das Modell deutlich stärker daneben liegt (da der RMSE größere Fehler stärker bestraft).

Fazit zur Modellgüte:

Obwohl das Modell klar identifiziert hat, dass der Stress Level der wichtigste Faktor ist, reicht diese Information (zusammen mit den anderen Features) nicht aus, um die Zielvariable präzise vorherzusagen.

Mögliche Gründe für die schwache Performance:

1. Es fehlen wichtige Einflussfaktoren (Features), die nicht im Datensatz enthalten sind.
2. Der Zusammenhang zwischen den Merkmalen ist sehr komplex oder verrauscht.
3. Die Datenmenge ist möglicherweise zu gering für ein präzises Random Forest Modell.

### 7.6 Modellstabilität (Cross Validation)

Zur Bewertung der Stabilität des Modells wird eine **k-fold Cross Validation** durchgeführt.
Die Ergebnisse sollen zeigen,  
- wie gut sich Schlafqualität aus Lifestyle-Variablen vorhersagen lässt,
- welche Faktoren den größten Einfluss besitzen und ob
- komplexe Modelle einen Mehrwert gegenüber linearen Modellen bieten.

In [46]:
# Modelle nur für CV
lr_cv = LinearRegression()
rf_cv = RandomForestRegressor(random_state=42, n_estimators=300)

# Cross-Validation
cv_scores_lr = cross_val_score(
    lr_cv,
    X_train_scaled_df,
    y_train_reg,
    cv=5,
    scoring="r2"
)

cv_scores_reg_rf = cross_val_score(
    rf_cv,
    X_train_scaled_df,
    y_train_reg,
    cv=5,
    scoring="r2"
)

# Ergebnisse
cv_compare_df = pd.DataFrame({
    "Model": ["Linear Regression"] * len(cv_scores_lr) + ["Random Forest"] * len(cv_scores_reg_rf),
    "Fold": list(range(1, 6)) + list(range(1, 6)),
    "R2": list(cv_scores_lr) + list(cv_scores_reg_rf)
})

display(
    cv_compare_df.style
    .format({"R2": "{:.3f}"})
    .hide(axis="index")
)

Model,Fold,R2
Linear Regression,1,0.486
Linear Regression,2,0.482
Linear Regression,3,0.480
Linear Regression,4,0.519
Linear Regression,5,0.484
Random Forest,1,0.432
Random Forest,2,0.447
Random Forest,3,0.408
Random Forest,4,0.526
Random Forest,5,0.485


In [47]:
import plotly.express as px

fig = px.box(
    cv_compare_df,
    x="Model",
    y="R2",
    color="Model",
    points="all",
    template="simple_white",
    title="Cross-Validation Vergleich – Regression auf Sleep Quality"
)

fig.update_layout(
    title_x=0.5,
    width=850,
    height=450,
    xaxis_title="",
    yaxis_title="R²",
    legend_title_text=""
)

fig.show()

In [48]:
cv_summary = (
    cv_compare_df
    .groupby("Model")["R2"]
    .agg(["mean", "std", "min", "max"])
    .reset_index()
    .rename(columns={
        "mean": "Mean R²",
        "std": "Std R²",
        "min": "Min R²",
        "max": "Max R²"
    })
)

display(
    cv_summary.style
    .format({
        "Mean R²": "{:.3f}",
        "Std R²": "{:.3f}",
        "Min R²": "{:.3f}",
        "Max R²": "{:.3f}"
    })
    .hide(axis="index")
)

Model,Mean R²,Std R²,Min R²,Max R²
Linear Regression,0.490,0.016,0.480,0.519
Random Forest,0.460,0.046,0.408,0.526


**Interpretation der Cross-Validation**

Die Cross-Validation vergleicht die Stabilität der Modelle über mehrere Trainings-/Testaufteilungen.

Ein stabiles Modell zeigt:

- einen hohen mittleren R²-Wert
- eine geringe Streuung zwischen den Folds

Der Vergleich zwischen der **Linearen Regression** und dem **Random Forest** zeigt deutliche Unterschiede in ihrer Stabilität und Vorhersagekraft:

- Vorhersagequalität (Mean R²):  
Die Lineare Regression erzielt mit einem durchschnittlichen von 0,490 ein besseres Ergebnis als der Random Forest (0,460). Das bedeutet, sie kann die Varianz der Schlafqualität im Durchschnitt besser erklären.
- Stabilität (Varianz):
    - Die Lineare Regression ist deutlich stabiler. Die Boxplot ist schmal, die Standardabweichung ist mit 0,016 gering, und die Ergebnisse der einzelnen Folds liegen nah beieinander (0,480 bis 0,519).
    - Der Random Forest zeigt eine viel größere Streuung. Die Box ist deutlich höher, die Standardabweichung ist mit 0,046 fast dreimal so hoch, und die Ergebnisse schwanken stark zwischen den Folds (0,408 bis 0,526).

Fazit:
Für diesen Datensatz ist die **Lineare Regression** die bessere Wahl. Sie liefert nicht nur im Durchschnitt bessere Vorhersagen, sondern ist vor allem **konsistenter**. Der Random Forest scheint "überfordert" zu sein oder leidet unter Overfitting in einzelnen Folds, was die weite Streuung der Ergebnisse erklärt.

### 7.7 Vorhersagefehler (Residual Plot)
Nach der Kreuzvalidierung ist ein Residual Plot der nächste Schritt. Während das Bestimmtheitsmaß R² ausdrückt, wie viel Varianz erklärt wird, zeigt ein Residual Plot, wo das Modell Fehler macht.  

Ein Residuum ist der Unterschied zwischen dem tatsächlich gemessenen Wert und dem vom Modell vorhergesagten Wert eines einzelnen Datenpunktes.

In [49]:
residual_df = pd.DataFrame({
    "Predicted": pred_lr,
    "Residual": y_test_reg - pred_lr
})

fig = px.scatter(
    residual_df,
    x="Predicted",
    y="Residual",
    trendline="lowess",
    template="simple_white",
    opacity=0.65,
    title="Residual Plot – Linear Regression"
)

# Bereich bestimmen
x_min = residual_df["Predicted"].min()
x_max = residual_df["Predicted"].max()

for trace in fig.data:

# Residual-Punkte
    if trace.mode == "markers":
        trace.marker.size = 7
        trace.marker.color = "rgba(80,120,200,0.6)"
        trace.name = "Residuals"
        trace.showlegend = True

# LOWESS Linie
    if trace.mode == "lines":
        trace.line.color = "#d62728"
        trace.line.width = 3
        trace.name = "LOWESS Trend"
        trace.showlegend = True

# Nulllinie als Scatter
fig.add_trace(
    go.Scatter(
        x=[x_min, x_max],
        y=[0, 0],
        mode="lines",
        line=dict(color="black", dash="dash", width=3),
        name="Zero Residual Line",
        showlegend=True
    )
)

fig.update_layout(
    title_x=0.5,
    width=900,
    height=500,
    xaxis_title="Predicted Sleep Quality",
    yaxis_title="Residual (Actual − Predicted)",
    legend_title_text=""
)

fig.show()

**Interpretation**

- Residuals (blaue Punkte): zeigen die Vorhersagefehler je Beobachtung
- LOWESS Trend (rote Linie): zeigt, ob die Residuen systematisch von 0 abweichen
- Zero Residual Line (schwarz gestrichelt): Referenzlinie für perfekte Vorhersage

Insgesamt ein erscheint das Modell sehr solide, weißt  aber Nuancen auf:
- **Linearität**: Die rote LOWESS-Linie verläuft fast deckungsgleich mit der gestrichelten Null-Linie. Das bedeutet, dass der Zusammenhang zwischen den Variablen weitgehend linear ist und keine groben systematischen Fehler vorliegen.
- **Homoskedastizität (Varianzhomogenität)**: Die Streuung der blauen Punkte scheint über den gesamten Bereich der vorhergesagten Schlafqualität ("Predicted Sleep Quality") relativ gleichmäßig zu sein. Es gibt keine ausgeprägte "Trichterform", was bedeutet, dass die Vorhersagegenauigkeit bei hoher Schlafqualität ähnlich gut ist wie bei niedriger.
- **Ausreißer**: Es gibt ein paar Punkte, die deutlich weiter von der Null-Linie entfernt sind (z. B. bei einer Vorhersage von ca. 6 und einem Residuum von fast 4). Diese Ausreißer könnten einzelne Nächte oder Personen sein, bei denen das Modell die Schlafqualität deutlich unterschätzt hat.

## 8. Klassifikation – High Stress

In diesem Kapitel wird untersucht, wie zuverlässig sich **hohes Stressniveau**
(`stressHigh`) anhand der verfügbaren Lifestyle- und Gesundheitsmerkmale klassifizieren lässt.

Verglichen werden zwei Modelltypen:

- **Logistische Regression** als gut interpretierbare Baseline
- **Random Forest Classifier** als leistungsfähiger nichtlinearer Ansatz

Die Modelle werden anhand mehrerer Kennzahlen bewertet:

- Accuracy
- Precision
- Recall
- F1-Score
- ROC-AUC


### 8.1 Baseline-Modell (Logistische Regression)

Die logistische Regression dient als einfacher und interpretierbarer Referenzwert
für die Klassifikation von hohem Stress.

In [50]:
log_reg = LogisticRegression(
    random_state=42,
    max_iter=1000
)

log_reg.fit(X_train_scaled_df, y_train_clf)

pred_log = log_reg.predict(X_test_scaled_df)
pred_proba_log = log_reg.predict_proba(X_test_scaled_df)[:, 1]

pred_train_lr = lr.predict(X_train_scaled_df)

train_r2_lr = r2_score(y_train_reg, pred_train_lr)

acc_log = accuracy_score(y_test_clf, pred_log)
prec_log = precision_score(y_test_clf, pred_log)
rec_log = recall_score(y_test_clf, pred_log)
f1_log = f1_score(y_test_clf, pred_log)
auc_log = roc_auc_score(y_test_clf, pred_proba_log)

baseline_clf_results = pd.DataFrame({
    "Model": ["Logistic Regression"],
    "Accuracy": [acc_log],
    "Precision": [prec_log],
    "Recall": [rec_log],
    "F1 Score": [f1_log],
    "ROC-AUC": [auc_log]
})

display(
    baseline_clf_results
    .style
    .format({
        "Accuracy": "{:.3f}",
        "Precision": "{:.3f}",
        "Recall": "{:.3f}",
        "F1 Score": "{:.3f}",
        "ROC-AUC": "{:.3f}"
    })
    .hide(axis="index")
)

Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
Logistic Regression,0.990,1.000,0.937,0.967,1.000


### 8.2 Advanced-Modell (Random Forest Classifier)

Der Random Forest Classifier kann nichtlineare Zusammenhänge und Interaktionen
zwischen Variablen flexibler abbilden als lineare Modelle.

In [51]:
rf_clf = RandomForestClassifier(
    random_state=42,
    n_estimators=50, #war 300
    min_samples_leaf=3
)

rf_clf.fit(X_train_scaled_df, y_train_clf)

pred_rf_clf = rf_clf.predict(X_test_scaled_df)
pred_proba_rf = rf_clf.predict_proba(X_test_scaled_df)[:, 1]

acc_rf = accuracy_score(y_test_clf, pred_rf_clf)
prec_rf = precision_score(y_test_clf, pred_rf_clf)
rec_rf = recall_score(y_test_clf, pred_rf_clf)
f1_rf = f1_score(y_test_clf, pred_rf_clf)
auc_rf = roc_auc_score(y_test_clf, pred_proba_rf)

rf_clf_results = pd.DataFrame({
    "Model": ["Random Forest"],
    "Accuracy": [acc_rf],
    "Precision": [prec_rf],
    "Recall": [rec_rf],
    "F1 Score": [f1_rf],
    "ROC-AUC": [auc_rf]
})

display(
    rf_clf_results
    .style
    .format({
        "Accuracy": "{:.3f}",
        "Precision": "{:.3f}",
        "Recall": "{:.3f}",
        "F1 Score": "{:.3f}",
        "ROC-AUC": "{:.3f}"
    })
    .hide(axis="index")
)

Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
Random Forest,1.000,1.000,1.000,1.000,1.000


### 8.3 Modellvergleich

Die Modelle werden direkt anhand ihrer Klassifikationsgüte verglichen.

In [52]:
clf_compare = pd.concat([baseline_clf_results, rf_clf_results], ignore_index=True)

display(
    clf_compare
    .style
    .format({
        "Accuracy": "{:.3f}",
        "Precision": "{:.3f}",
        "Recall": "{:.3f}",
        "F1 Score": "{:.3f}",
        "ROC-AUC": "{:.3f}"
    })
    .hide(axis="index")
)

Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
Logistic Regression,0.990,1.000,0.937,0.967,1.000
Random Forest,1.000,1.000,1.000,1.000,1.000


In [53]:
plot_df = clf_compare.melt(
    id_vars="Model",
    var_name="Metric",
    value_name="Value"
)

fig = px.bar(
    plot_df,
    x="Metric",
    y="Value",
    color="Model",
    barmode="group",
    template="simple_white",
    title="Modellvergleich – Klassifikation von High Stress"
)

fig.update_traces(
    width=0.2,  # Deutlich schmalere Balken (vorher default ~0.8-0.9)
    marker_line_width=0.5,
    marker_line_color="white"
)

fig.update_layout(
    title_x=0.5,
    width=950,
    height=450,
    xaxis_title="",
    yaxis_title="Wert",
    legend_title_text="",
    bargap=0.5,  # Abstand zwischen Balkengruppen
    bargroupgap=0.5  # Abstand innerhalb der Gruppe
)

fig.show()

**Interpretation**  

- **Überlegene Leistung des Random Forest**: Der Random Forest erreicht in allen fünf Metriken (Accuracy, Precision, Recall, F1 Score und ROC-AUC) nahezu den Maximalwert von 1,0. Das deutet darauf hin, dass dieses Modell die Testdaten fast perfekt klassifiziert.
- **Starke Logistische Regression mit kleinen Schwächen**: Die Logistische Regression liefert ebenfalls sehr gute Ergebnisse, fällt aber im Vergleich leicht ab, insbesondere beim Recall (ca. 0,93) und folglich auch beim F1 Score (ca. 0,96).
- **Bedeutung für die Praxis**: Während beide Modelle extrem präzise sind (Precision ist bei beiden fast 1,0), erkennt der Random Forest einen größeren Anteil der tatsächlichen Stress-Fälle (Recall). In einem medizinischen oder psychologischen Kontext ist ein hoher Recall oft entscheidend, um niemanden zu übersehen.

### 8.4 Confusion Matrix

Die Confusion Matrix zeigt, wie gut das Modell hohe Stressfälle und Nicht-Fälle unterscheidet.

In [54]:
cm = confusion_matrix(y_test_clf, pred_rf_clf)

cm_df = pd.DataFrame(
    cm,
    index=["Actual 0", "Actual 1"],
    columns=["Predicted 0", "Predicted 1"]
)

fig = px.imshow(
    cm_df,
    text_auto=True,
    aspect="auto",
    template="simple_white",
    title="Confusion Matrix – Random Forest"
)

fig.update_layout(
    title_x=0.5,
    width=600,
    height=500
)

fig.show()

**Interpretation**
Diese Confusion Matrix für ein Random Forest Modell zeigt ein perfektes Ergebnis für den vorliegenden Datensatz. Das Modell hat jede einzelne Instanz korrekt klassifiziert.  
Hier ist die detaillierte Aufschlüsselung der Werte:
- True Negatives (TN): 337  
Das Modell hat 337 Fälle korrekt als "0" vorhergesagt, die auch tatsächlich "0" waren.
- True Positives (TP): 63  
Das Modell hat 63 Fälle korrekt als "1" vorhergesagt, die auch tatsächlich "1" waren.
- False Positives (FP): 0
Es gab keine Fälle, in denen fälschlicherweise eine "1" vorhergesagt wurde.
- False Negatives (FN): 0
Es gab keine Fälle, in denen eine tatsächliche "1" übersehen wurde.


Wenn Accuracy, Precision und Recall 100% erreichen, dann könnte das auf **Overfitting** hindeuten oder darauf, dass Testdaten beim Training verwendet wurden (Data Leakage), was hier aber ausgeschlossen werden kann. Es könnte auch sein, dass ein einzelnes Feature fast die gesamte Vorhersagekraft hat. Aus Zeitgründen wurde Anpassung der Modellparameter und die erforderliche weitere Untersuchung abgebrochen.

### 8.5 Receiver Operating Characteristic (ROC)
Die ROC-Kurve ist ein wichtiges Werkzeug, um die Leistung von binären Klassifikationsmodellen zu bewerten.  
Sie soll die die **Trennschärfe des Modells** über alle möglichen Schwellenwerte zeigen.

In [55]:
fpr_log, tpr_log, _ = roc_curve(y_test_clf, pred_proba_log)
fpr_rf, tpr_rf, _ = roc_curve(y_test_clf, pred_proba_rf)

fig = go.Figure()

# Logistic Regression
fig.add_trace(
    go.Scatter(
        x=fpr_log,
        y=tpr_log,
        mode="lines",
        name="Logistic Regression",
        line=dict(color="#1f77b4", width=3)
    )
)

# Random Forest
fig.add_trace(
    go.Scatter(
        x=fpr_rf,
        y=tpr_rf,
        mode="lines",
        name="Random Forest",
        line=dict(color="#d62728", width=3, dash="dot")
    )
)

# Zufallsbaseline
fig.add_trace(
    go.Scatter(
        x=[0, 1],
        y=[0, 1],
        mode="lines",
        name="Random Baseline",
        line=dict(color="black", dash="dash", width=2)
    )
)

fig.update_layout(
    template="simple_white",
    title="ROC Curve – High Stress Classification",
    title_x=0.5,
    width=850,
    height=500,
    xaxis_title="False Positive Rate",
    yaxis_title="True Positive Rate",
    legend_title_text=""
)

fig.show()

**Interpretation**
Die ROC-Kurve (Receiver Operating Characteristic)  ist ein wichtiges Werkzeug, um die Leistung von binären Klassifikationsmodellen zu bewerten.

1. Die Modellleistung  
Sowohl die **Logistische Regression** (blaue Linie) als auch der **Random Forest** (rote gepunktete Linie) zeigen eine nahezu perfekte Performance.  
- Die Kurven verlaufen fast rechtwinklig. Das bedeutet, dass beide Modelle eine True Positive Rate (Sensitivität) von nahezu 1,0 erreichen, während die False Positive Rate bei fast 0 bleibt.
- Die AUC (Area Under the Curve) liegt extrem nah bei 1,0. Das wäre ein theoretisch perfektes Modell.
2. Vergleich zum Zufall
- Die gestrichelte diagonale Linie stellt die **Random Baseline** dar (AUC = 0,5). Sie zeigt, wie gut ein Modell wäre, das rein zufällig rät.
- Da beide Modelle weit über dieser Linie liegen, sind sie statistisch signifikant besser als der Zufall.
3. Kritische Einordnung
- **Data Leakage**: Informationen aus der Zielvariable könnten unbeabsichtigt in den Trainingsdaten enthalten sein.
- **Überanpassung (Overfitting)**: Das Modell hat die Trainingsdaten eventuell auswendig gelernt, anstatt allgemeine Muster zu erkennen.
- **Zu kleiner Datensatz**: Bei sehr wenigen Datenpunkten ist es für ein Modell leicht, eine perfekte Trennung zu finden.

### 8.6 Feature Importance

Zur explorativen Interpretation wird die relative Bedeutung der Features im Random-Forest-Modell betrachtet.

In [56]:
imp_clf = pd.Series(
    rf_clf.feature_importances_,
    index=base_features
).sort_values()

imp_clf.index = imp_clf.index.map(lambda c: labels.get(c, c))

fig = px.bar(
    imp_clf.reset_index().rename(columns={"index": "Feature", 0: "Importance"}),
    x="Importance",
    y="Feature",
    orientation="h",
    template="simple_white",
    title="Feature Importance – Random Forest (Classification)"
)

fig.update_layout(
    title_x=0.5,
    width=900,
    height=500
)

fig.show()

display(
    imp_clf.to_frame("Importance")
    .style
    .format({"Importance": "{:.3f}"})
)

,Importance
Age,0.009
Sugar Intake (g/day),0.010
Alcohol (drinks/week),0.012
Physical Activity (min/day),0.014
Sleep Duration (hours),0.049
Heart Rate (bpm),0.069
Stress Level (1–10),0.836


### 8.7 Modellstabilität (Cross-Validation)

Die Cross-Validation bewertet die Stabilität der Klassifikationsleistung über mehrere Datenaufteilungen.

In [57]:
cv_scores_log = cross_val_score(
    log_reg,
    X_train_scaled_df,
    y_train_clf,
    cv=5,
    scoring="roc_auc"
)

cv_scores_clf_rf = cross_val_score(
    rf_clf,
    X_train_scaled_df,
    y_train_clf,
    cv=5,
    scoring="roc_auc"
)

cv_compare_clf = pd.DataFrame({
    "Model": ["Logistic Regression"] * len(cv_scores_log) + ["Random Forest"] * len(cv_scores_clf_rf),
    "Fold": list(range(1, 6)) + list(range(1, 6)),
    "ROC-AUC": list(cv_scores_log) + list(cv_scores_clf_rf)
})

display(
    cv_compare_clf
    .style
    .format({"ROC-AUC": "{:.3f}"})
    .hide(axis="index")
)

Model,Fold,ROC-AUC
Logistic Regression,1,1.000
Logistic Regression,2,0.999
Logistic Regression,3,1.000
Logistic Regression,4,0.999
Logistic Regression,5,1.000
Random Forest,1,1.000
Random Forest,2,1.000
Random Forest,3,1.000
Random Forest,4,1.000
Random Forest,5,1.000


In [58]:
fig = px.box(
    cv_compare_clf,
    x="Model",
    y="ROC-AUC",
    color="Model",
    points="all",
    template="simple_white",
    title="Cross-Validation Vergleich – High Stress Classification"
)

fig.update_layout(
    title_x=0.5,
    width=850,
    height=450,
    xaxis_title="",
    yaxis_title="ROC-AUC",
    legend_title_text=""
)

fig.show()

**Interpretation der Cross-Validation**

Die Ergebnisse zeigen,

- wie zuverlässig sich hohes Stressniveau klassifizieren lässt,
- ob ein nichtlineares Modell einen Mehrwert gegenüber der logistischen Regression bietet,
- und welche Variablen den größten Beitrag zur Vorhersage leisten.

Modellvergleich (ROC-AUC)

- **Random Forest** (Orange): Das Modell erreicht fast durchgehend einen perfekten ROC-AUC von 1.0. Das bedeutet, es kann fehlerfrei zwischen "High Stress" und anderen Zuständen unterscheiden. Die Punkte links neben dem Balken zeigen, dass die Ergebnisse über verschiedene Cross-Validation-Durchläufe hinweg extrem stabil sind.
- **Logistische Regression** (Blau): Die Performance des Modells ist ebenfalls sehr hoch (über 0.998), zeigt aber eine deutlich größere Varianz. Die Box und die einzelnen Datenpunkte verdeutlichen, dass die Ergebnisse je nach Daten-Split stärker schwanken.

### 8.8 Precision-Recall-Kurve
Für Klassifikation ist der diagnostische Gegenpart zum Residual Plot die Precision-Recall-Kurve und eine Threshold-Analyse. 
Besonders relevant sind dabei:

- **Recall (Sensitivität)**: wenn möglichst viele High-Stress-Fälle erkannt werden sollen (FN vermeiden).
- **Precision (Genauigkeit)**: um Fehlalarme zu minimieren (FP vermeiden).
- **ROC-AUC**, wenn die allgemeine Trennschärfe des Modells über verschiedene Schwellenwerte hinweg bewertet werden soll.

In [59]:
precision_log, recall_log, _ = precision_recall_curve(y_test_clf, pred_proba_log)
precision_rf, recall_rf, _ = precision_recall_curve(y_test_clf, pred_proba_rf)

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=recall_log,
        y=precision_log,
        mode="lines",
        name="Logistic Regression"
    )
)

fig.add_trace(
    go.Scatter(
        x=recall_rf,
        y=precision_rf,
        mode="lines",
        name="Random Forest"
    )
)

fig.update_layout(
    template="simple_white",
    title="Precision-Recall Curve – High Stress Classification",
    title_x=0.5,
    width=850,
    height=500,
    xaxis_title="Recall",
    yaxis_title="Precision",
    legend_title_text=""
)

fig.show()

**Interpretation**

Die zeigt den Zusammenhang zwischen Precision (Genauigkeit) und Recall (Trefferquote), wenn der Klassifikationsschwellenwert variiert. 
Auffällig ist, dass beide Kurven fast identisch verlaufen und sich beinahe perfekt in der oberen rechten Ecke (Punkt 1,1) befinden.  
Das Modell kann eine Recall-Rate von fast 100% erreichen, ohne dass die Precision nennenswert sinkt. 

Neben einem zu untersuchenden Data Leakage könnte **fehlende Komplexität** ein möglicher Grund sein. Eventuell ist die Trennung zwischen "Stress" und "kein Stress" im vorliegenden Datensatz zu eindeutig/einfach.

### 8.9 Threshold-Analyse
Der Klassifikationsschwellenwert beeinflusst die Balance zwischen Precision und Recall.
- niedriger Schwellenwert → mehr erkannte High-Stress-Fälle (höherer Recall)
- höherer Schwellenwert → weniger Fehlalarme (höhere Precision)

Die Analyse hilft dabei, einen Schwellenwert zu wählen, der zum Anwendungsziel passt.

In [60]:
precision, recall, thresholds = precision_recall_curve(y_test_clf, pred_proba_rf)

threshold_df = pd.DataFrame({
    "Threshold": thresholds,
    "Precision": precision[:-1],
    "Recall": recall[:-1]
})

fig = px.line(
    threshold_df,
    x="Threshold",
    y=["Precision","Recall"],
    template="simple_white",
    title="Threshold Analyse – Random Forest"
)

fig.update_layout(
    title_x=0.5,
    width=850,
    height=450,
    xaxis_title="Classification Threshold",
    yaxis_title="Score",
    legend_title_text=""
)

fig.show()

**Interpretation**

Auffällig an diesem spezifischen Modell ist das extrem breite Plateau, auf dem sowohl Precision als auch Recall nahe bei 1,0 liegen.
- **Niedriger Schwellenwert (< 0,2)**: Die Precision ist anfangs niedrig. Das Modell ist "vorsichtig" und schlägt bei fast allem Alarm. Dadurch wird zwar jeder Stressfall gefunden (Recall = 1), aber es gibt viele Fehlalarme.
- Der **"Sweet Spot" (ca. 0,2 bis 0,85)**: In diesem Bereich sind beide Kurven auf ihrem Maximum. Das Modell arbeitet hier extrem präzise und erkennt gleichzeitig fast alle relevanten Fälle.
- **Hoher Schwellenwert (> 0,9)**: Der Recall bricht ein. Das Modell wird zu "wählerisch". Es schlägt nur noch Alarm, wenn es sich absolut sicher ist, wodurch viele tatsächliche Stress-Fälle unentdeckt bleiben.

**Wahl des Schwellenwerts**
Da das Plateau so breit ist, ist das Modell sehr robust. Die Wahl hängt dennoch vom Einsatzzweck ab:  
- **Prävention**: Ein Schwellenwert um 0,3 bis 0,4 stellt sicher, dass kein Stressfall übersehen wird (Recall bleibt bei 1), während die Precision bereits ihr Maximum erreicht hat.
- **Ressourcenschonung**: Ein Schwellenwert um 0,7 bis 0,8 vermeidet Fehlalarme fast vollständig, nimmt aber ein minimal höheres Risiko in Kauf, einen Grenzfall zu übersehen.

### 8.10 Festlegung eines Entscheidungs-Schwellwertes (F1-Score)

In [61]:
thresholds = np.linspace(0.05, 0.95, 100)

precision_scores = []
recall_scores = []
f1_scores = []

for t in thresholds:

    preds = (pred_proba_rf >= t).astype(int)

    precision_scores.append(
        precision_score(y_test_clf, preds)
    )

    recall_scores.append(
        recall_score(y_test_clf, preds)
    )

    f1_scores.append(
        f1_score(y_test_clf, preds)
    )

# optimaler Threshold nach F1
best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]

best_threshold

np.float64(0.21363636363636362)

In [62]:
df_plot = pd.DataFrame({
    "Threshold": thresholds,
    "F1-Score": f1_scores,
    "Precision": precision_scores,
    "Recall": recall_scores
})

df_melted = df_plot.melt(
    id_vars=["Threshold"],
    value_vars=["F1-Score", "Precision", "Recall"],
    var_name="Metrik",
    value_name="Wert"
)

# 2. Diagramm erstellen
fig = px.line(
    df_melted,
    x="Threshold",
    y="Wert",
    color="Metrik",
    template="simple_white",
    title="Optimierung des Klassifikations-Thresholds",
    labels={"Wert": "Score"}
)

# 3. Optimale Schwelle markieren
fig.add_vline(
    x=best_threshold,
    line_dash="dash",
    line_color="black",
    annotation_text=f"Optimum: {best_threshold:.2f}"
)

fig.update_layout(
    title_x=0.5,
    width=850,
    height=500,
    legend_title_text=""
)

fig.show()

**Interpretation**

- **Plateau-Effekt**: Der F1-Score erreicht sein Maximum (nahe 1.0) in einem breiten Bereich von Thresholds zwischen etwa 0.2 und 0.8. Das deutet darauf hin, dass das Modell in diesem Bereich sehr stabil ist.
- **Bester Threshold**: Der optimale Schwellwert liegt bei ca. 0.21. Dies ist der Punkt, an dem der maximale F1-Score zum ersten Mal stabil erreicht wird.
- **Abfall an den Rändern**:
    - Unter 0.15 sinkt der F1-Score rapide.
    - Über 0.85 sinkt er ebenfalls stark.


### 8.11 Finale Confusion Matrix

Jetzt wird das Modell mit dem **optimierten Threshold** angewendet.

In [63]:
pred_rf_opt = (pred_proba_rf >= best_threshold).astype(int)

cm = confusion_matrix(y_test_clf, pred_rf_opt)

cm_df = pd.DataFrame(
    cm,
    index=["Actual Low Stress", "Actual High Stress"],
    columns=["Predicted Low", "Predicted High"]
)

fig = px.imshow(
    cm_df,
    text_auto=True,
    aspect="auto",
    template="simple_white",
    title="Finale Confusion Matrix – Random Forest"
)

fig.update_layout(
    title_x=0.5,
    width=600,
    height=500
)

fig.show()

**Interpretation**  
Wie erwartet ergeben sich mit dem Modell und dem optimierten Schwellwert keine Änderungen.

## 9. Modellierungsfazit – Performance Overview

Zum Abschluss von Notebook 2 werden die wichtigsten Ergebnisse der Modellierung
kompakt zusammengeführt.

Die Übersicht stellt die finalen Modelle für

- **Regression:** Vorhersage der Schlafqualität
- **Klassifikation:** Identifikation von High Stress

in einer gemeinsamen Grafik dar.

Damit wird sichtbar:

- welches Modell innerhalb der jeweiligen Aufgabe besser performt
- wie groß der Unterschied zwischen Baseline und Advanced-Modell ist
- welche Modellfamilie sich insgesamt als leistungsstärker erweist

In [64]:
# --- Regressionsergebnisse vorbereiten
reg_compare = model_compare.copy()
reg_compare["Task"] = "Regression"
reg_compare = reg_compare.rename(columns={
    "Root Mean Squared Error": "RMSE",
    "Mean Absolute Error": "MAE",
    "Coefficient of Determination (R²)": "R²"
})

# --- Klassifikationsergebnisse vorbereiten
clf_overview = clf_compare.copy()
clf_overview["Task"] = "Classification"
clf_overview = clf_overview.rename(columns={
    "Accuracy": "Accuracy",
    "Precision": "Precision",
    "Recall": "Recall",
    "F1 Score": "F1",
    "ROC-AUC": "ROC-AUC"
})

# --- Long-Format
reg_long = reg_compare.melt(
    id_vars=["Model", "Task"],
    var_name="Metric",
    value_name="Value"
)

clf_long = clf_overview.melt(
    id_vars=["Model", "Task"],
    var_name="Metric",
    value_name="Value"
)

perf_long = pd.concat([reg_long, clf_long], ignore_index=True)

# --- Visualisierung
fig = px.bar(
    perf_long,
    x="Metric",
    y="Value",
    color="Model",
    facet_row="Task",
    barmode="group",
    template="simple_white",
    title="Model Performance Overview"
)

fig.update_layout(
    title_x=0.5,
    width=1100,
    height=700,
    legend_title_text=""
)

fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))

fig.show()

**Interpretation**

Die Modellierungsphase liefert damit zwei finale Analysepfade:

- ein Regressionsmodell zur Vorhersage der Schlafqualität
- ein Klassifikationsmodell zur Identifikation von High-Stress-Fällen

Vergleich der Regressionsmodelle:  
Hier werden die **Lineare Regression** (blau) und **Random Forest** (orange) verglichen:

- **Fehlermerkmale (RMSE & MAE)**: Beide Modelle zeigen sehr ähnliche Werte. Überraschenderweise liegt der RMSE über 1, was auf größere Abweichungen bei einzelnen Vorhersagen hindeutet.
- **Bestimmtheitsmaß (R²)**: Mit Werten um 0,4 erklären die Modelle nur etwa 40 % der Varianz in der Schlafqualität. Das deutet darauf hin, dass die Vorhersagekraft moderat ist und noch wichtige Einflussfaktoren fehlen könnten.

Vergleich der Klassifikationsmodelle:  
Hier treten **Random Forest** (orange) und **Logistische Regression** (grün) gegeneinander an:

- **Überlegene Performance**: Der Random Forest dominiert fast alle Metriken (Accuracy, Precision, Recall, F1, ROC-AUC) mit Werten nahe 1,0. Er identifiziert High-Stress-Fälle nahezu perfekt.
- **Logistische Regression**: Diese zeigt ebenfalls eine sehr gute Leistung (Werte meist über 0,9), fällt aber besonders beim Recall etwas gegenüber dem Random Forest ab. Das bedeutet, dass sie ein paar Stress-Fälle eher übersieht als das Random-Forest-Modell.

Während die Klassifikation von Stress hervorragend funktioniert, bleibt die präzise Vorhersage der Schlafqualität (Regression) eine Herausforderung. Der Random Forest scheint insgesamt das robustere Modell für beide Aufgabenstellungen zu sein.  

Im folgenden Notebook werden diese Modelle nun:

- systematisch evaluiert,
- im Hinblick auf Stabilität und Aussagekraft verglichen
- und zur Prüfung der zuvor formulierten Hypothesen herangezogen.

## 10. Datenexport

In [65]:
#Export der Modellergebnisse zur Weiterverwendung in Notebook 3

model_compare.to_csv("data/regression_results.csv", index=False)
clf_compare.to_csv("data/classification_results.csv", index=False)
pd.Series(cv_scores_lr).to_csv("data/cv_scores_lr.csv", index=False)
pd.Series(cv_scores_reg_rf).to_csv("data/cv_scores_regression.csv", index=False)
pd.Series(cv_scores_clf_rf).to_csv("data/cv_scores_classification.csv", index=False)

# --- Regression Train Performance
pred_train_rf = rf.predict(X_train_scaled_df)
train_r2_rf = r2_score(y_train_reg, pred_train_rf)

# --- Classification Train Performance
pred_train_proba_rf = rf_clf.predict_proba(X_train_scaled_df)[:,1]
train_auc_rf = roc_auc_score(y_train_clf, pred_train_proba_rf)

test_r2_lr = r2_lr

performance_summary = pd.DataFrame({
    "metric":[
        "train_r2_lr",
        "test_r2_lr",
        "train_r2_rf",
        "test_r2_rf",
        "train_auc_rf",
        "test_auc_rf"
    ],
    "value":[
        train_r2_lr,
        test_r2_lr,
        train_r2_rf,
        r2_rf,
        train_auc_rf,
        auc_rf
    ]
})
performance_summary.to_csv("data/model_performance_summary.csv", index=False)

feature_importance = pd.DataFrame({
    "feature": base_features,
    "importance": rf.feature_importances_
})
feature_importance.to_csv("data/feature_importance_rf.csv", index=False)

cluster_data = df_cluster[["cluster"]].copy()
cluster_data["sleepQuality"] = df_clean["sleepQuality"].values
cluster_data["stressHigh"] = df_clean["stressHigh"].values
cluster_data["riskScore"] = df_clean["riskScore"].values
cluster_summary = (
    cluster_data
    .groupby("cluster")
    .agg({
        "sleepQuality": "mean",
        "stressHigh": "mean",
        "riskScore": "mean"
    })
    .reset_index()
)
cluster_summary["cluster_label"] = cluster_summary["cluster"].map(
    lambda x: f"Cluster {int(x)}"
)

cluster_summary.to_csv("data/cluster_summary.csv", index=False)
cluster_data.to_csv("data/cluster_data.csv", index=False)

In [66]:
# dashboard data export

dashboard_df = df_clean.copy()

dashboard_df["cluster"] = df_cluster["cluster"].values
dashboard_df["cluster_label"] = dashboard_df["cluster"].map(cluster_labels)

# Spalten für dash auswählen
dashboard_columns = [

    "age",
    "gender",
    "married",
    "smoking",
    "bmiCategory",
    "healthRisk",
    "riskScore",

    "stress",
    "sleepDuration",
    "sleepQuality",
    "heartRate",

    "activity",
    "steps",
    "alcohol",
    "sugar",

    "shortSleep",
    "lifestyleRisk",
    "stressSleepMismatch",
    "stressHigh",

    "cluster",
    "cluster_label"
]

dashboard_columns = [c for c in dashboard_columns if c in dashboard_df.columns]
dashboard_df = dashboard_df[dashboard_columns]


dashboard_df.to_csv("data/dashboard_data.csv", index=False)
dashboard_df.to_parquet("data/dashboard_data.parquet")

print("Dashboard dataset exported")
print("Shape:", dashboard_df.shape)

Dashboard dataset exported
Shape: (2000, 21)


[🔝 Nach oben](#top)